
# Student Stress Prediction — MASTER Notebook E0–E7

## Reliable Student Stress Prediction under Population Shift

This notebook contains the **complete experimental pipeline E0–E7** in execution order:

- **E0** — Rigorous data audit
- **E1** — Reproducible IID baseline
- **E2** — Leave-One-Student-Type-Out population shift
- **E3** — Calibration under population shift
- **E4** — ERM vs GroupDRO + threshold adaptation
- **E5** — Conformal prediction + selective abstention
- **E6** — Capacity-constrained intervention policy
- **E7** — Robustness, statistical inference, and publication-ready synthesis

## How to run on your computer

1. Put this notebook and `student-lifestyle-and-stress-dataset.csv` in the **same folder**.  
   Alternatively, put the CSV inside a subfolder named `data/`.
2. Install the packages listed in `requirements_student_stress.txt`.
3. Restart the Python kernel.
4. Use **Run All** from top to bottom.

Each experiment writes its results to a separate local folder:

- `student_stress_e0_e2_outputs/`
- `student_stress_e3_outputs/`
- `student_stress_e4_outputs/`
- `student_stress_e5_outputs/`
- `student_stress_e6_outputs/`
- `student_stress_e7_outputs/`

E7 intentionally reads the outputs created by E0–E6, so **do not skip earlier sections when running the master notebook from a clean directory**.

## Reproducibility principle

All preprocessing, calibration, conformal calibration, threshold learning, and policy construction are restricted to the appropriate training/source data. Held-out LOSO populations remain evaluation-only except where a cell is explicitly labelled as a **non-deployable oracle diagnostic**.

## Interpretation boundary

This public dataset is treated as a methodological benchmark. Results describe predictive behavior and robustness; they do not establish causal, clinical, or population-prevalence conclusions.


In [ ]:

# MASTER environment check
from pathlib import Path
import sys

MASTER_DATA_CANDIDATES = [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
]
MASTER_DATA_PATH = next(
    (p for p in MASTER_DATA_CANDIDATES if p.exists()),
    None
)

if MASTER_DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Place 'student-lifestyle-and-stress-dataset.csv' "
        "beside this notebook or inside a ./data folder."
    )

print("Master notebook dataset:", MASTER_DATA_PATH.resolve())
print("Python:", sys.version.split()[0])
print("Ready to run E0–E7.")


# ========== E0–E2 — Audit, IID Baseline, LOSO Shift ==========

The following cells are preserved from the validated standalone experiment notebook.


# Student Stress Prediction: E0–E2 Reliability Experiments

This notebook establishes the methodological foundation for a reliability-centered student-stress prediction paper.

## Experiments
- **E0:** rigorous data audit.
- **E1A:** IID repeated stratified CV using all predictors.
- **E1B:** IID repeated stratified CV excluding `Student_Type`.
- **E2:** Leave-One-Student-Type-Out (LOSO), with `Student_Type` used only to define the unseen domain.

The primary transportability comparison is **E1B → E2** because both use the same predictor set.

> Associations in this benchmark dataset are predictive, not causal.


In [ ]:

from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    log_loss, matthews_corrcoef, confusion_matrix
)

SEED = 42
TARGET = "Stress_Level"
GROUP = "Student_Type"
N_SPLITS = 5
N_REPEATS = 2

DATA_CANDIDATES = [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Set DATA_PATH to the CSV location.")

RESULT_DIR = Path("../outputs/E0_E2")
RESULT_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("Dataset:", DATA_PATH.resolve())


In [ ]:

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


# E0 — Rigorous Data Audit

In [ ]:

missing = pd.DataFrame({
    "missing_n": df.isna().sum(),
    "missing_pct": 100 * df.isna().mean(),
}).sort_values("missing_pct", ascending=False)

predictors = [c for c in df.columns if c != TARGET]
duplicates = pd.DataFrame({
    "check": [
        "extra_full_duplicate_rows",
        "extra_duplicate_predictor_rows",
        "conflicting_labels_for_duplicate_predictors",
    ],
    "count": [
        int(df.duplicated().sum()),
        int(df.duplicated(subset=predictors).sum()),
        int((df.groupby(predictors, dropna=False)[TARGET].nunique() > 1).sum()),
    ],
})

display(missing)
display(duplicates)
missing.to_csv(RESULT_DIR / "e0_missingness.csv")
duplicates.to_csv(RESULT_DIR / "e0_duplicates.csv", index=False)


In [ ]:

numeric_cols = df.select_dtypes(include=np.number).columns
ranges = pd.DataFrame({
    "min": df[numeric_cols].min(),
    "median": df[numeric_cols].median(),
    "mean": df[numeric_cols].mean(),
    "max": df[numeric_cols].max(),
    "missing_n": df[numeric_cols].isna().sum(),
    "unique_n": df[numeric_cols].nunique(),
})
display(ranges)
ranges.to_csv(RESULT_DIR / "e0_numeric_ranges.csv")

rules = {
    "Sleep_Hours_outside_0_24":
        df["Sleep_Hours"].notna() & ~df["Sleep_Hours"].between(0, 24),
    "Study_Hours_outside_0_24":
        df["Study_Hours"].notna() & ~df["Study_Hours"].between(0, 24),
    "Social_Media_Hours_outside_0_24":
        df["Social_Media_Hours"].notna() & ~df["Social_Media_Hours"].between(0, 24),
    "Attendance_outside_0_100":
        df["Attendance"].notna() & ~df["Attendance"].between(0, 100),
    "Exam_Pressure_outside_1_10":
        df["Exam_Pressure"].notna() & ~df["Exam_Pressure"].between(1, 10),
    "Family_Support_outside_1_10":
        df["Family_Support"].notna() & ~df["Family_Support"].between(1, 10),
    "Month_outside_1_12":
        df["Month"].notna() & ~df["Month"].between(1, 12),
}

validity = pd.DataFrame([
    {"rule": name, "invalid_n": int(mask.sum()), "invalid_pct": 100 * mask.mean()}
    for name, mask in rules.items()
])
display(validity)
validity.to_csv(RESULT_DIR / "e0_validity_flags.csv", index=False)


In [ ]:

student_prev = (
    df.assign(Student_Type_Audit=df[GROUP].fillna("MISSING"))
      .groupby("Student_Type_Audit")[TARGET]
      .agg(n="size", high_stress_n="sum", high_stress_rate="mean")
      .reset_index()
)
student_prev["high_stress_pct"] = 100 * student_prev["high_stress_rate"]

month_prev = (
    df.groupby("Month", dropna=False)[TARGET]
      .agg(n="size", high_stress_n="sum", high_stress_rate="mean")
      .reset_index()
)
month_prev["high_stress_pct"] = 100 * month_prev["high_stress_rate"]

display(student_prev)
display(month_prev)

student_prev.to_csv(RESULT_DIR / "e0_prevalence_by_student_type.csv", index=False)
month_prev.to_csv(RESULT_DIR / "e0_prevalence_by_month.csv", index=False)

ax = student_prev.set_index("Student_Type_Audit")["high_stress_pct"].plot(
    kind="bar", figsize=(7, 4)
)
ax.set_ylabel("High-stress prevalence (%)")
ax.set_xlabel("")
ax.set_title("E0 — High-stress prevalence by Student_Type")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(RESULT_DIR / "fig_e0_prevalence_student_type.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

signal_cols = [
    "Sleep_Hours", "Study_Hours", "Social_Media_Hours",
    "Attendance", "Exam_Pressure", "Family_Support", "Month"
]
signal_rows = []
for col in signal_cols:
    tmp = df[[col, TARGET]].dropna()
    corr = tmp[col].corr(tmp[TARGET])
    signal_rows.append({
        "feature": col,
        "n_complete": len(tmp),
        "point_biserial_corr": corr,
        "abs_corr": abs(corr),
    })

signal_audit = (
    pd.DataFrame(signal_rows)
      .sort_values("abs_corr", ascending=False)
)
display(signal_audit)
signal_audit.to_csv(RESULT_DIR / "e0_numeric_signal_audit.csv", index=False)



# Leakage-safe baseline pipeline

For E0–E2 we intentionally use **Logistic Regression as the primary model**.  
The immediate question is whether the validation protocol exposes population-shift risk; model-complexity comparisons are deferred.

- Numeric: median imputation + standardization.
- Categorical: most-frequent imputation + one-hot encoding.
- `Month` is categorical.
- Threshold-dependent metrics use 0.50 only as an initial diagnostic.


In [ ]:

NUMERIC_FEATURES = [
    "Sleep_Hours", "Study_Hours", "Social_Media_Hours",
    "Attendance", "Exam_Pressure", "Family_Support"
]
ALL_FEATURES = NUMERIC_FEATURES + ["Student_Type", "Month"]
GROUPBLIND_FEATURES = NUMERIC_FEATURES + ["Month"]

def make_lr_pipeline(categorical_cols):
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_cols),
    ], sparse_threshold=0)

    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=SEED,
        )),
    ])

def classification_metrics(y_true, prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob),
        "pr_auc": average_precision_score(y_true, prob),
        "brier": brier_score_loss(y_true, prob),
        "nll": log_loss(y_true, prob, labels=[0, 1]),
    }

METRICS = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "specificity", "f1", "mcc", "roc_auc", "pr_auc", "brier", "nll"
]


# E1 — Reproducible IID Baseline

In [ ]:

def run_iid(features, categorical_cols, feature_set_name):
    X = df[features].copy()
    y = df[TARGET].astype(int).copy()

    cv = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=SEED,
    )

    rows = []
    for split_id, (tr, te) in enumerate(cv.split(X, y), start=1):
        model = make_lr_pipeline(categorical_cols)
        model.fit(X.iloc[tr], y.iloc[tr])
        prob = model.predict_proba(X.iloc[te])[:, 1]

        rows.append({
            "feature_set": feature_set_name,
            "repeat": (split_id - 1) // N_SPLITS + 1,
            "fold": (split_id - 1) % N_SPLITS + 1,
            "n_train": len(tr),
            "n_test": len(te),
            **classification_metrics(y.iloc[te], prob),
        })
    return pd.DataFrame(rows)

e1a = run_iid(ALL_FEATURES, ["Student_Type", "Month"], "E1A_ALL")
e1b = run_iid(GROUPBLIND_FEATURES, ["Month"], "E1B_GROUPBLIND")
e1 = pd.concat([e1a, e1b], ignore_index=True)

e1_summary = e1.groupby("feature_set")[METRICS].agg(["mean", "std"])
display(e1_summary)
e1.to_csv(RESULT_DIR / "e1_fold_metrics.csv", index=False)
e1_summary.to_csv(RESULT_DIR / "e1_summary_mean_sd.csv")


## E1B subgroup OOF diagnostic

In [ ]:

X = df[GROUPBLIND_FEATURES].copy()
y = df[TARGET].astype(int).copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_prob = np.full(len(df), np.nan)

for tr, te in cv.split(X, y):
    model = make_lr_pipeline(["Month"])
    model.fit(X.iloc[tr], y.iloc[tr])
    oof_prob[te] = model.predict_proba(X.iloc[te])[:, 1]

oof = pd.DataFrame({
    "y_true": y.values,
    "prob": oof_prob,
    "Student_Type": df[GROUP].fillna("MISSING").values,
})

subgroup_rows = []
for grp, part in oof.groupby("Student_Type"):
    subgroup_rows.append({
        "Student_Type": grp,
        "n": len(part),
        "prevalence": part["y_true"].mean(),
        **classification_metrics(part["y_true"], part["prob"]),
    })

e1_subgroup = pd.DataFrame(subgroup_rows)
display(e1_subgroup[
    ["Student_Type", "n", "prevalence", "roc_auc", "pr_auc",
     "recall", "specificity", "brier", "nll"]
])
e1_subgroup.to_csv(
    RESULT_DIR / "e1b_lr_oof_subgroup_metrics.csv", index=False
)



# E2 — Leave-One-Student-Type-Out

`Student_Type` is excluded from predictors and used only to define the unseen domain.  
Rows with missing `Student_Type` are excluded because their domain membership is undefined.


In [ ]:

e2_df = df[df[GROUP].notna()].copy()
groups = sorted(e2_df[GROUP].unique())

rows = []
for heldout in groups:
    train_mask = e2_df[GROUP] != heldout
    test_mask = e2_df[GROUP] == heldout

    X_train = e2_df.loc[train_mask, GROUPBLIND_FEATURES]
    y_train = e2_df.loc[train_mask, TARGET].astype(int)
    X_test = e2_df.loc[test_mask, GROUPBLIND_FEATURES]
    y_test = e2_df.loc[test_mask, TARGET].astype(int)

    model = make_lr_pipeline(["Month"])
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]

    rows.append({
        "heldout_group": heldout,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "train_prevalence": y_train.mean(),
        "test_prevalence": y_test.mean(),
        **classification_metrics(y_test, prob),
    })

e2 = pd.DataFrame(rows)
display(e2[
    ["heldout_group", "n_test", "test_prevalence",
     "roc_auc", "pr_auc", "recall", "specificity", "brier", "nll"]
])
e2.to_csv(RESULT_DIR / "e2_loso_metrics.csv", index=False)


In [ ]:

weights = e2["n_test"] / e2["n_test"].sum()

higher = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "specificity", "f1", "mcc", "roc_auc", "pr_auc"
]
lower = ["brier", "nll"]

iid_mean = e1b[METRICS].mean()
shift_rows = []

for metric in METRICS:
    loso_weighted = np.sum(e2[metric] * weights)
    loso_worst = e2[metric].min() if metric in higher else e2[metric].max()

    weighted_worsening = (
        iid_mean[metric] - loso_weighted
        if metric in higher
        else loso_weighted - iid_mean[metric]
    )
    worst_worsening = (
        iid_mean[metric] - loso_worst
        if metric in higher
        else loso_worst - iid_mean[metric]
    )

    shift_rows.append({
        "metric": metric,
        "iid_groupblind_mean": iid_mean[metric],
        "loso_weighted": loso_weighted,
        "loso_worst_group": loso_worst,
        "weighted_worsening": weighted_worsening,
        "worst_group_worsening": worst_worsening,
    })

shift_penalty = pd.DataFrame(shift_rows)
display(shift_penalty[
    shift_penalty["metric"].isin(
        ["roc_auc", "pr_auc", "recall", "specificity", "brier", "nll"]
    )
])
shift_penalty.to_csv(
    RESULT_DIR / "e1b_to_e2_shift_penalty.csv", index=False
)


In [ ]:

ax = e2.set_index("heldout_group")[["roc_auc", "recall", "specificity"]].plot(
    kind="bar", figsize=(8, 4)
)
ax.set_ylabel("Metric value")
ax.set_xlabel("Held-out Student_Type")
ax.set_title("E2 — LOSO discrimination and threshold behavior")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(RESULT_DIR / "fig_e2_loso_core_metrics.png", dpi=200, bbox_inches="tight")
plt.show()



# Decision Gate for the Next Experiments

Proceed to calibration-under-shift, GroupDRO, and conformal prediction when at least one of the following is observed:

1. LOSO is worse than E1B IID.
2. Worst-group performance is substantially below weighted-average performance.
3. ROC-AUC remains similar while recall/specificity varies materially across populations.
4. Brier/NLL deteriorate under held-out-domain evaluation.

A small AUC drop **does not invalidate** the hypothesis if operational or probabilistic reliability differs by population.


In [ ]:

manifest = {
    "dataset_rows": len(df),
    "dataset_columns": df.shape[1],
    "seed": SEED,
    "iid_protocol": f"RepeatedStratifiedKFold({N_SPLITS} folds x {N_REPEATS} repeats)",
    "primary_model": "LogisticRegression",
    "e1a_features": ALL_FEATURES,
    "e1b_e2_features": GROUPBLIND_FEATURES,
    "e2_missing_group_rows_excluded": int(df[GROUP].isna().sum()),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}
with open(RESULT_DIR / "run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved files:")
for p in sorted(RESULT_DIR.glob("*")):
    print("-", p)


# ========== E3 — Calibration Under Population Shift ==========

The following cells are preserved from the validated standalone experiment notebook.

# E3 — Calibration Under Population Shift

This notebook evaluates **uncalibrated → sigmoid → isotonic** probability estimates under:

1. **E3-IID:** repeated stratified outer CV;
2. **E3-LOSO:** Leave-One-Student-Type-Out population shift.

The primary model is the **group-blind Logistic Regression** carried forward from E1B/E2.

## Core design

For every outer training set, a 3-fold inner CV generates **out-of-fold base probabilities**.  
Sigmoid and isotonic calibrators are fitted only on those inner OOF predictions. The base model is then refitted on the complete outer-training data and applied to the untouched outer test set. For LOSO, the held-out `Student_Type` is never used to fit either the classifier or the calibrator.

## Actual E3-IID global results from this run

| method       |   roc_auc |   pr_auc |   brier |   brier_skill_score |     nll |   ece_uniform |   ece_quantile |   mce_uniform |   mce_quantile |   calibration_intercept |   calibration_slope |
|:-------------|----------:|---------:|--------:|--------------------:|--------:|--------------:|---------------:|--------------:|---------------:|------------------------:|--------------------:|
| isotonic     |   0.86494 |  0.74942 | 0.12983 |             0.38164 | 0.40921 |       0.00211 |        0.00191 |       0.00591 |        0.00375 |                -0.00766 |             0.9878  |
| sigmoid      |   0.86575 |  0.75192 | 0.12975 |             0.38202 | 0.40866 |       0.01014 |        0.0096  |       0.02332 |        0.01981 |                 0.00184 |             1.00322 |
| uncalibrated |   0.86575 |  0.75193 | 0.12974 |             0.38206 | 0.40865 |       0.00966 |        0.00941 |       0.02182 |        0.01958 |                -0.00121 |             0.99856 |

## Actual E3-LOSO results from this run

| heldout_group   | method       |   brier |   brier_skill_score |     nll |   ece_uniform |   ece_quantile |   mce_uniform |   mce_quantile |   calibration_intercept |   calibration_slope |
|:----------------|:-------------|--------:|--------------------:|--------:|--------------:|---------------:|--------------:|---------------:|------------------------:|--------------------:|
| college         | uncalibrated | 0.13428 |             0.37743 | 0.42112 |       0.00886 |        0.00972 |       0.02867 |        0.01999 |                 0.02288 |             0.99758 |
| college         | sigmoid      | 0.13429 |             0.37738 | 0.42112 |       0.00912 |        0.00985 |       0.02527 |        0.0211  |                 0.02471 |             1.00213 |
| college         | isotonic     | 0.13453 |             0.37628 | 0.42605 |       0.01029 |        0.00521 |       0.02599 |        0.01925 |                 0.00041 |             0.95539 |
| school          | uncalibrated | 0.10989 |             0.38529 | 0.35641 |       0.02034 |        0.01811 |       0.06292 |        0.04813 |                -0.06718 |             0.98611 |
| school          | sigmoid      | 0.10991 |             0.38519 | 0.35641 |       0.01886 |        0.01843 |       0.05401 |        0.04906 |                -0.06257 |             0.99387 |
| school          | isotonic     | 0.10993 |             0.38505 | 0.35581 |       0.01248 |        0.01139 |       0.04828 |        0.04181 |                -0.04087 |             1.01648 |
| working_student | uncalibrated | 0.1451  |             0.41167 | 0.44781 |       0.0134  |        0.01168 |       0.04441 |        0.02209 |                 0.01326 |             0.97626 |
| working_student | sigmoid      | 0.14509 |             0.4117  | 0.4478  |       0.01354 |        0.01151 |       0.0455  |        0.02105 |                 0.01739 |             0.98023 |
| working_student | isotonic     | 0.14549 |             0.41011 | 0.44912 |       0.01883 |        0.0161  |       0.03744 |        0.03461 |                 0.00088 |             0.94513 |

## Weighted / worst-group LOSO summary

| method       |   roc_auc_weighted |   pr_auc_weighted |   brier_weighted |   bss_weighted |   nll_weighted |   ece_uniform_weighted |   ece_quantile_weighted |   mce_uniform_worst |   mce_quantile_worst |   worst_group_abs_intercept |   worst_group_abs_slope_error |
|:-------------|-------------------:|------------------:|-----------------:|---------------:|---------------:|-----------------------:|------------------------:|--------------------:|---------------------:|----------------------------:|------------------------------:|
| isotonic     |            0.85906 |           0.72473 |          0.1306  |        0.3852  |        0.41321 |                0.01253 |                 0.00892 |             0.04828 |              0.04181 |                     0.04087 |                       0.05487 |
| sigmoid      |            0.85965 |           0.73566 |          0.13039 |        0.38616 |        0.41037 |                0.01242 |                 0.01231 |             0.05401 |              0.04906 |                     0.06257 |                       0.01977 |
| uncalibrated |            0.85965 |           0.73566 |          0.13038 |        0.38621 |        0.41037 |                0.01262 |                 0.01219 |             0.06292 |              0.04813 |                     0.06718 |                       0.02374 |

> Associations and probability estimates are benchmark-model results, not causal or clinical claims.


In [ ]:
from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from scipy.special import expit, logit
from scipy.optimize import minimize

SEED=42
TARGET="Stress_Level"
GROUP="Student_Type"
N_BINS=10
EPS=1e-6
DATA_PATH=next((p for p in [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv")
] if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Edit DATA_PATH.")
RESULT_DIR=Path("../outputs/E3")
RESULT_DIR.mkdir(exist_ok=True)
df=pd.read_csv(DATA_PATH)
print(df.shape)


In [ ]:
NUMERIC_FEATURES=[
    "Sleep_Hours","Study_Hours","Social_Media_Hours",
    "Attendance","Exam_Pressure","Family_Support"
]
CATEGORICAL_FEATURES=["Month"]
FEATURES=NUMERIC_FEATURES+CATEGORICAL_FEATURES

def make_base_lr():
    prep=ColumnTransformer([
        ("num",Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scaler",StandardScaler()),
        ]),NUMERIC_FEATURES),
        ("cat",Pipeline([
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False)),
        ]),CATEGORICAL_FEATURES),
    ],sparse_threshold=0)
    return Pipeline([
        ("prep",prep),
        ("model",LogisticRegression(max_iter=2000,solver="lbfgs",random_state=SEED)),
    ])

def inner_oof_base_probs(X,y,n_splits=3):
    cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=SEED)
    out=np.full(len(y),np.nan)
    for tr,va in cv.split(X,y):
        m=make_base_lr()
        m.fit(X.iloc[tr],y.iloc[tr])
        out[va]=m.predict_proba(X.iloc[va])[:,1]
    return out

def fit_three(X_train,y_train,X_test):
    Xr=X_train.reset_index(drop=True)
    yr=y_train.reset_index(drop=True)
    inner_p=inner_oof_base_probs(Xr,yr,3)

    xlog=logit(np.clip(inner_p,EPS,1-EPS)).reshape(-1,1)
    sig=LogisticRegression(C=1e6,solver="lbfgs",max_iter=2000)
    sig.fit(xlog,yr)

    iso=IsotonicRegression(out_of_bounds="clip",y_min=0,y_max=1)
    iso.fit(inner_p,yr)

    base=make_base_lr()
    base.fit(X_train,y_train)
    raw=base.predict_proba(X_test)[:,1]

    sigmoid=sig.predict_proba(logit(np.clip(raw,EPS,1-EPS)).reshape(-1,1))[:,1]
    isotonic=np.clip(iso.predict(raw),EPS,1-EPS)

    return {"uncalibrated":raw,"sigmoid":sigmoid,"isotonic":isotonic}


In [ ]:
def calibration_bins(y,p,n_bins=10,strategy="uniform"):
    y=np.asarray(y).astype(int)
    p=np.clip(np.asarray(p,float),0,1)
    edges=(np.linspace(0,1,n_bins+1) if strategy=="uniform"
           else np.unique(np.quantile(p,np.linspace(0,1,n_bins+1))))
    if len(edges)<2:
        edges=np.array([0.,1.])
    else:
        edges[0]=min(edges[0],0); edges[-1]=max(edges[-1],1)
    ids=np.digitize(p,edges[1:-1],right=True)
    rows=[]
    for b in range(len(edges)-1):
        mask=ids==b
        if mask.sum():
            mp=float(p[mask].mean()); er=float(y[mask].mean())
            rows.append({"bin":b+1,"n":int(mask.sum()),"mean_pred":mp,
                         "event_rate":er,"abs_gap":abs(er-mp)})
    return pd.DataFrame(rows)

def ece_mce(y,p,n_bins=10,strategy="uniform"):
    bins=calibration_bins(y,p,n_bins,strategy)
    ece=float(((bins["n"]/bins["n"].sum())*bins["abs_gap"]).sum())
    return ece,float(bins["abs_gap"].max())

def calib_intercept_slope(y,p):
    y=np.asarray(y).astype(float)
    p=np.clip(np.asarray(p,float),EPS,1-EPS)
    x=logit(p)
    def obj(beta):
        q=np.clip(expit(beta[0]+beta[1]*x),EPS,1-EPS)
        return -np.mean(y*np.log(q)+(1-y)*np.log(1-q))
    res=minimize(obj,[0.,1.],method="BFGS")
    return float(res.x[0]),float(res.x[1])

def metrics(y,p,reference_prob):
    y=np.asarray(y).astype(int)
    p=np.clip(np.asarray(p,float),EPS,1-EPS)
    ref=np.asarray(reference_prob,float)
    if ref.ndim==0:
        ref=np.full(len(y),float(ref))
    bs=brier_score_loss(y,p)
    bs_ref=float(np.mean((y-ref)**2))
    eu,mu=ece_mce(y,p,N_BINS,"uniform")
    eq,mq=ece_mce(y,p,N_BINS,"quantile")
    ci,cs=calib_intercept_slope(y,p)
    return {
        "roc_auc":roc_auc_score(y,p),
        "pr_auc":average_precision_score(y,p),
        "brier":bs,
        "brier_reference":bs_ref,
        "brier_skill_score":1-bs/bs_ref,
        "nll":log_loss(y,p,labels=[0,1]),
        "ece_uniform":eu,"ece_quantile":eq,
        "mce_uniform":mu,"mce_quantile":mq,
        "calibration_intercept":ci,"calibration_slope":cs,
    }


## E3-IID — repeated outer CV with inner OOF calibration

In [ ]:
X=df[FEATURES].copy()
y=df[TARGET].astype(int).copy()
outer=RepeatedStratifiedKFold(n_splits=5,n_repeats=2,random_state=SEED)

fold_rows=[]
pred_rows=[]
for sid,(tr,te) in enumerate(outer.split(X,y),1):
    rep=(sid-1)//5+1
    fold=(sid-1)%5+1
    Xtr,Xte=X.iloc[tr],X.iloc[te]
    ytr,yte=y.iloc[tr],y.iloc[te]
    prev=float(ytr.mean())

    for method,p in fit_three(Xtr,ytr,Xte).items():
        fold_rows.append({
            "method":method,"repeat":rep,"fold":fold,
            "n_train":len(tr),"n_test":len(te),"train_prevalence":prev,
            **metrics(yte,p,prev)
        })
        for idx,yt,pr in zip(te,yte.to_numpy(),p):
            pred_rows.append({
                "row_id":int(idx),"repeat":rep,"fold":fold,
                "method":method,"y_true":int(yt),"prob":float(pr),
                "reference_prob":prev,
                "Student_Type":df.loc[idx,GROUP] if pd.notna(df.loc[idx,GROUP]) else "MISSING"
            })

iid_fold=pd.DataFrame(fold_rows)
iid_predictions=pd.DataFrame(pred_rows)
display(iid_fold.groupby("method").mean(numeric_only=True))


In [ ]:
iid_oof=(iid_predictions
    .groupby(["row_id","method","y_true","Student_Type"],as_index=False)
    .agg(prob=("prob","mean"),reference_prob=("reference_prob","mean")))

global_rows=[]
subgroup_rows=[]
for method,part in iid_oof.groupby("method"):
    global_rows.append({"method":method,"n":len(part),
                        "prevalence":part.y_true.mean(),
                        **metrics(part.y_true,part.prob,part.reference_prob)})

for (method,grp),part in iid_oof.groupby(["method","Student_Type"]):
    subgroup_rows.append({"method":method,"Student_Type":grp,"n":len(part),
                          "prevalence":part.y_true.mean(),
                          **metrics(part.y_true,part.prob,part.reference_prob)})

iid_global=pd.DataFrame(global_rows)
iid_subgroup=pd.DataFrame(subgroup_rows)
display(iid_global)
display(iid_subgroup)


## E3-LOSO — calibration on training populations only

In [ ]:
e2=df[df[GROUP].notna()].copy()
rows=[]
preds=[]
for heldout in sorted(e2[GROUP].unique()):
    train=e2[e2[GROUP]!=heldout]
    test=e2[e2[GROUP]==heldout]
    Xtr=train[FEATURES]; ytr=train[TARGET].astype(int)
    Xte=test[FEATURES]; yte=test[TARGET].astype(int)
    prev=float(ytr.mean())

    for method,p in fit_three(Xtr,ytr,Xte).items():
        rows.append({
            "heldout_group":heldout,"method":method,
            "n_train":len(train),"n_test":len(test),
            "train_prevalence":prev,"test_prevalence":float(yte.mean()),
            **metrics(yte,p,prev)
        })
        for idx,yt,pr in zip(test.index,yte.to_numpy(),p):
            preds.append({"row_id":int(idx),"heldout_group":heldout,
                          "method":method,"y_true":int(yt),"prob":float(pr),
                          "reference_prob":prev})

loso=pd.DataFrame(rows)
loso_predictions=pd.DataFrame(preds)
display(loso)


In [ ]:
def plot_reliability(data,title,strategy="quantile"):
    fig,ax=plt.subplots(figsize=(6,5))
    ax.plot([0,1],[0,1],linestyle="--",linewidth=1,label="Perfect calibration")
    for method in ["uncalibrated","sigmoid","isotonic"]:
        part=data[data["method"]==method]
        bins=calibration_bins(part.y_true,part.prob,N_BINS,strategy)
        ax.plot(bins.mean_pred,bins.event_rate,marker="o",label=method)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed high-stress rate")
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    plt.show()

plot_reliability(iid_oof,"E3-IID — Global reliability")

for grp in sorted(loso_predictions.heldout_group.unique()):
    plot_reliability(
        loso_predictions[loso_predictions.heldout_group==grp],
        f"E3-LOSO — Unseen {grp}"
    )


## Interpretation rule

- Lower is better: **Brier, NLL, ECE, MCE**.
- Higher is better: **Brier Skill Score**.
- Calibration intercept should approach **0**.
- Calibration slope should approach **1**.
- Calibration must be judged jointly across metrics and held-out groups.
- Isotonic can improve ECE while worsening NLL/Brier or discrimination because it is more flexible and can create ties.


# ========== E4 — ERM vs GroupDRO + Threshold Adaptation ==========

The following cells are preserved from the validated standalone experiment notebook.


# E4 — ERM vs GroupDRO + Threshold Adaptation

## Scientific objective

E0–E3 showed that population shift produced only a modest loss in rank discrimination, while threshold-dependent sensitivity differed substantially across `Student_Type`. E3 further showed that post-hoc calibration did not consistently solve this problem.

E4 therefore asks:

> **Is the population gap primarily a learning-objective problem, or can it be explained and reduced by decision-threshold adaptation?**

## Models

Both models use the **same group-blind linear logistic architecture** and the same leakage-safe preprocessing.

1. **ERM Logistic** — minimizes average binary cross-entropy.
2. **GroupDRO Logistic** — minimizes an adaptive weighted combination of group-specific losses, emphasizing the currently worst training groups.

`Student_Type` is **never used as a predictor**. For GroupDRO it is used only as a training-domain label.

## Main deployment policies

- **ERM@0.50**
- **ERM@TrainRecall80** — threshold selected only from 3-fold OOF predictions within the LOSO training populations.
- **GroupDRO@0.50**
- **GroupDRO@TrainRecall80**

Target:

\[
Recall \ge 0.80
\]

The training-OOF threshold is selected as the **largest threshold** that still satisfies the recall constraint, which preferentially preserves specificity.

## Oracle diagnostic

For scientific attribution only, E4 also reports a **HeldoutOracleRecall80** threshold chosen using labels from the held-out population itself.

This oracle is **not deployable and must never be presented as a valid external-validation method**. It estimates the upper bound of what threshold adaptation alone could recover if the target-population labels were known.

## Primary evaluation

Leave-One-Student-Type-Out (LOSO):
- train on two populations,
- test on the third unseen population,
- rotate across school, college, and working_student.

This directly extends E2.


In [ ]:

from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    log_loss, matthews_corrcoef, confusion_matrix, precision_recall_curve
)
from scipy.special import expit

SEED = 42
TARGET = "Stress_Level"
GROUP = "Student_Type"
INNER_SPLITS = 3
TARGET_RECALL = 0.80

EPOCHS = 180
LEARNING_RATE = 0.035
L2 = 1e-4
DRO_ETA = 0.10

DATA_CANDIDATES = [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found. Edit DATA_PATH.")

RESULT_DIR = Path("../outputs/E4")
RESULT_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)

print("Python :", platform.python_version())
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("Dataset:", DATA_PATH.resolve())
print("Target recall:", TARGET_RECALL)
print("Inner threshold CV:", INNER_SPLITS)


In [ ]:

df = pd.read_csv(DATA_PATH)

NUMERIC_FEATURES = [
    "Sleep_Hours",
    "Study_Hours",
    "Social_Media_Hours",
    "Attendance",
    "Exam_Pressure",
    "Family_Support",
]
CATEGORICAL_FEATURES = ["Month"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

e4_df = df[df[GROUP].notna()].copy().reset_index(drop=False).rename(columns={"index": "original_row_id"})
groups = sorted(e4_df[GROUP].unique().tolist())

print("Full dataset:", df.shape)
print("E4 rows with known Student_Type:", len(e4_df))
print("Excluded missing Student_Type:", int(df[GROUP].isna().sum()))
print("Groups:", groups)



# Leakage-safe preprocessing

The preprocessor is fitted separately inside every inner training fold and again on the full LOSO training set.

- numeric: median imputation + standardization;
- `Month`: most-frequent imputation + one-hot encoding;
- `Student_Type`: excluded from predictors.


In [ ]:

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CATEGORICAL_FEATURES),
    ], sparse_threshold=0)

def fit_transform_train_test(train_df, test_df):
    prep = make_preprocessor()
    X_train = prep.fit_transform(train_df[FEATURES]).astype(np.float64)
    X_test = prep.transform(test_df[FEATURES]).astype(np.float64)
    return X_train, X_test, prep



# Common Logistic Model: ERM and GroupDRO

A custom NumPy implementation is used so both objectives share exactly the same model class, initialization, optimizer, regularization, and preprocessing.

### ERM

\[
L_{ERM} = \frac{1}{N}\sum_i \ell_i
\]

### GroupDRO

For training group \(g\):

\[
L_g = \frac{1}{N_g}\sum_{i \in g}\ell_i
\]

Adaptive group weights are updated by:

\[
q_g \leftarrow
\frac{q_g\exp(\eta L_g)}
{\sum_j q_j\exp(\eta L_j)}
\]

and the training objective becomes:

\[
L_{DRO}=\sum_g q_gL_g
\]

Thus groups with persistently larger training loss receive increasing optimization weight.


In [ ]:

def binary_logloss_from_logits(y, z):
    return np.logaddexp(0.0, z) - y * z

def fit_logistic_objective(
    X, y, group_labels=None, objective="ERM",
    epochs=EPOCHS, lr=LEARNING_RATE, l2=L2,
    dro_eta=DRO_ETA
):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n, d = X.shape

    w = np.zeros(d, dtype=np.float64)
    prevalence = np.clip(y.mean(), 1e-4, 1 - 1e-4)
    b = float(np.log(prevalence / (1 - prevalence)))

    # Adam state
    mw = np.zeros_like(w)
    vw = np.zeros_like(w)
    mb = 0.0
    vb = 0.0
    beta1, beta2, eps_adam = 0.9, 0.999, 1e-8

    group_info = None
    if objective == "GroupDRO":
        g = np.asarray(group_labels).astype(str)
        unique_groups = np.unique(g)
        group_indices = {grp: np.where(g == grp)[0] for grp in unique_groups}
        q = np.ones(len(unique_groups), dtype=np.float64) / len(unique_groups)

    for epoch in range(1, epochs + 1):
        z = X @ w + b
        p = expit(np.clip(z, -35, 35))
        residual = p - y

        if objective == "ERM":
            sample_weights = np.full(n, 1.0 / n)
        elif objective == "GroupDRO":
            losses = binary_logloss_from_logits(y, z)
            group_losses = np.array([
                losses[group_indices[grp]].mean()
                for grp in unique_groups
            ])

            # Stable multiplicative update
            log_q = np.log(np.clip(q, 1e-15, None)) + dro_eta * group_losses
            log_q -= log_q.max()
            q = np.exp(log_q)
            q /= q.sum()

            sample_weights = np.zeros(n, dtype=np.float64)
            for qg, grp in zip(q, unique_groups):
                idx = group_indices[grp]
                sample_weights[idx] = qg / len(idx)
        else:
            raise ValueError("objective must be ERM or GroupDRO")

        grad_w = X.T @ (sample_weights * residual) + l2 * w
        grad_b = float(np.sum(sample_weights * residual))

        # Adam
        mw = beta1 * mw + (1 - beta1) * grad_w
        vw = beta2 * vw + (1 - beta2) * (grad_w ** 2)
        mb = beta1 * mb + (1 - beta1) * grad_b
        vb = beta2 * vb + (1 - beta2) * (grad_b ** 2)

        mw_hat = mw / (1 - beta1 ** epoch)
        vw_hat = vw / (1 - beta2 ** epoch)
        mb_hat = mb / (1 - beta1 ** epoch)
        vb_hat = vb / (1 - beta2 ** epoch)

        w -= lr * mw_hat / (np.sqrt(vw_hat) + eps_adam)
        b -= lr * mb_hat / (np.sqrt(vb_hat) + eps_adam)

    if objective == "GroupDRO":
        group_info = {
            grp: float(qg) for grp, qg in zip(unique_groups, q)
        }

    return {"w": w, "b": b, "group_weights": group_info}

def predict_proba_linear(model, X):
    z = np.asarray(X) @ model["w"] + model["b"]
    return expit(np.clip(z, -35, 35))


# Metrics and threshold selection

In [ ]:

def probability_metrics(y_true, prob):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(prob, dtype=float)
    return {
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p),
        "brier": brier_score_loss(y, p),
        "nll": log_loss(y, p, labels=[0, 1]),
    }

def decision_metrics(y_true, prob, threshold):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(prob, dtype=float)
    pred = (p >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, pred, zero_division=0),
        "mcc": matthews_corrcoef(y, pred),
        "intervention_rate": float(pred.mean()),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

def select_recall_constrained_threshold(y_true, prob, target_recall=TARGET_RECALL):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(prob, dtype=float)

    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        return 0.5

    valid = recall[:-1] >= target_recall
    if not np.any(valid):
        return 0.0

    # Largest threshold satisfying the sensitivity constraint.
    return float(np.max(thresholds[valid]))



# Inner OOF threshold learning

For each LOSO training set and each objective:

1. split the **training populations only** into 3 stratified folds;
2. fit preprocessing and model on each inner training fold;
3. predict its validation fold;
4. pool OOF probabilities;
5. choose the global threshold satisfying `Recall ≥ 0.80`.

No held-out-population observation participates in this threshold selection.


In [ ]:

def training_oof_predictions(train_df, objective, seed=SEED):
    y = train_df[TARGET].astype(int).to_numpy()
    g = train_df[GROUP].astype(str).to_numpy()

    cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=seed,
    )

    oof = np.full(len(train_df), np.nan)

    for fold, (tr, va) in enumerate(cv.split(train_df, y), start=1):
        inner_train = train_df.iloc[tr]
        inner_valid = train_df.iloc[va]

        X_tr, X_va, _ = fit_transform_train_test(inner_train, inner_valid)

        model = fit_logistic_objective(
            X_tr,
            inner_train[TARGET].astype(int).to_numpy(),
            group_labels=inner_train[GROUP].astype(str).to_numpy(),
            objective=objective,
        )
        oof[va] = predict_proba_linear(model, X_va)

    if np.isnan(oof).any():
        raise RuntimeError("Incomplete OOF predictions.")

    return oof


# E4 LOSO Experiment

In [ ]:

OBJECTIVES = ["ERM", "GroupDRO"]

model_rows = []
policy_rows = []
pred_rows = []
threshold_rows = []
dro_weight_rows = []

for heldout in groups:
    train_df = e4_df[e4_df[GROUP] != heldout].copy()
    test_df = e4_df[e4_df[GROUP] == heldout].copy()

    y_train = train_df[TARGET].astype(int).to_numpy()
    y_test = test_df[TARGET].astype(int).to_numpy()

    for objective in OBJECTIVES:
        # ---------- training-only threshold learning ----------
        train_oof_prob = training_oof_predictions(train_df, objective)
        train_threshold = select_recall_constrained_threshold(
            y_train, train_oof_prob, TARGET_RECALL
        )
        train_oof_policy = decision_metrics(
            y_train, train_oof_prob, train_threshold
        )

        # ---------- final model ----------
        X_train, X_test, prep = fit_transform_train_test(train_df, test_df)
        model = fit_logistic_objective(
            X_train,
            y_train,
            group_labels=train_df[GROUP].astype(str).to_numpy(),
            objective=objective,
        )
        test_prob = predict_proba_linear(model, X_test)

        # Non-deployable oracle threshold, for attribution only.
        oracle_threshold = select_recall_constrained_threshold(
            y_test, test_prob, TARGET_RECALL
        )

        threshold_rows.append({
            "heldout_group": heldout,
            "objective": objective,
            "train_oof_threshold_recall80": train_threshold,
            "train_oof_achieved_recall": train_oof_policy["recall"],
            "train_oof_specificity": train_oof_policy["specificity"],
            "heldout_oracle_threshold_recall80": oracle_threshold,
            "oracle_is_deployable": False,
        })

        model_rows.append({
            "heldout_group": heldout,
            "objective": objective,
            "n_train": len(train_df),
            "n_test": len(test_df),
            "train_prevalence": y_train.mean(),
            "test_prevalence": y_test.mean(),
            **probability_metrics(y_test, test_prob),
        })

        policies = {
            "Fixed0.50": (0.50, True),
            "TrainRecall80": (train_threshold, True),
            "HeldoutOracleRecall80": (oracle_threshold, False),
        }

        for policy, (threshold, deployable) in policies.items():
            policy_rows.append({
                "heldout_group": heldout,
                "objective": objective,
                "policy": policy,
                "deployable": deployable,
                "n_test": len(test_df),
                **decision_metrics(y_test, test_prob, threshold),
            })

        for row_id, yt, pr in zip(
            test_df["original_row_id"].to_numpy(),
            y_test,
            test_prob,
        ):
            pred_rows.append({
                "original_row_id": int(row_id),
                "heldout_group": heldout,
                "objective": objective,
                "y_true": int(yt),
                "prob": float(pr),
                "train_threshold_recall80": train_threshold,
                "oracle_threshold_recall80": oracle_threshold,
            })

        if objective == "GroupDRO":
            for train_group, weight in model["group_weights"].items():
                dro_weight_rows.append({
                    "heldout_group": heldout,
                    "training_group": train_group,
                    "final_group_weight": weight,
                })

model_metrics = pd.DataFrame(model_rows)
policy_metrics = pd.DataFrame(policy_rows)
predictions = pd.DataFrame(pred_rows)
thresholds = pd.DataFrame(threshold_rows)
dro_weights = pd.DataFrame(dro_weight_rows)

model_metrics.to_csv(RESULT_DIR / "e4_loso_probability_metrics.csv", index=False)
policy_metrics.to_csv(RESULT_DIR / "e4_loso_policy_metrics.csv", index=False)
predictions.to_csv(RESULT_DIR / "e4_loso_predictions.csv", index=False)
thresholds.to_csv(RESULT_DIR / "e4_thresholds.csv", index=False)
dro_weights.to_csv(RESULT_DIR / "e4_groupdro_final_group_weights.csv", index=False)

display(model_metrics.round(5))
display(thresholds.round(5))
display(dro_weights.round(5))


# Aggregate deployable-policy comparison

In [ ]:

deployable = policy_metrics[policy_metrics["deployable"]].copy()

aggregate_rows = []
for (objective, policy), part in deployable.groupby(["objective", "policy"]):
    weights = part["n_test"] / part["n_test"].sum()

    pooled_pred_parts = []
    for heldout in groups:
        pred_part = predictions[
            (predictions["objective"] == objective) &
            (predictions["heldout_group"] == heldout)
        ].copy()

        if policy == "Fixed0.50":
            threshold = 0.50
        elif policy == "TrainRecall80":
            threshold = float(
                thresholds[
                    (thresholds["objective"] == objective) &
                    (thresholds["heldout_group"] == heldout)
                ]["train_oof_threshold_recall80"].iloc[0]
            )
        else:
            continue

        pred_part["decision"] = (
            pred_part["prob"] >= threshold
        ).astype(int)
        pooled_pred_parts.append(pred_part)

    pooled = pd.concat(pooled_pred_parts, ignore_index=True)
    y_pool = pooled["y_true"].to_numpy()
    d_pool = pooled["decision"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(
        y_pool, d_pool, labels=[0, 1]
    ).ravel()

    aggregate_rows.append({
        "objective": objective,
        "policy": policy,
        "weighted_recall": np.sum(weights * part["recall"]),
        "worst_group_recall": part["recall"].min(),
        "best_group_recall": part["recall"].max(),
        "recall_gap": part["recall"].max() - part["recall"].min(),
        "weighted_specificity": np.sum(weights * part["specificity"]),
        "worst_group_specificity": part["specificity"].min(),
        "weighted_balanced_accuracy": np.sum(
            weights * part["balanced_accuracy"]
        ),
        "weighted_precision": np.sum(weights * part["precision"]),
        "weighted_intervention_rate": np.sum(
            weights * part["intervention_rate"]
        ),
        "pooled_recall": tp / (tp + fn),
        "pooled_specificity": tn / (tn + fp),
        "pooled_precision": tp / (tp + fp) if (tp + fp) else np.nan,
        "pooled_intervention_rate": d_pool.mean(),
        "pooled_false_negatives": int(fn),
        "pooled_false_positives": int(fp),
    })

aggregate = pd.DataFrame(aggregate_rows).sort_values(
    ["policy", "objective"]
)
display(aggregate.round(5))
aggregate.to_csv(
    RESULT_DIR / "e4_deployable_policy_aggregate.csv",
    index=False
)


# Attribution: how much can threshold adaptation recover?

In [ ]:

# Compare fixed, training-selected, and oracle recall for every objective/group.
attrib = policy_metrics.pivot_table(
    index=["heldout_group", "objective"],
    columns="policy",
    values=["threshold", "recall", "specificity", "intervention_rate"]
)
attrib.columns = [
    f"{metric}_{policy}" for metric, policy in attrib.columns
]
attrib = attrib.reset_index()

attrib["recall_gain_train_threshold_vs_fixed"] = (
    attrib["recall_TrainRecall80"] -
    attrib["recall_Fixed0.50"]
)
attrib["recall_gain_oracle_vs_fixed"] = (
    attrib["recall_HeldoutOracleRecall80"] -
    attrib["recall_Fixed0.50"]
)
attrib["specificity_cost_train_threshold_vs_fixed"] = (
    attrib["specificity_Fixed0.50"] -
    attrib["specificity_TrainRecall80"]
)
attrib["intervention_increase_train_threshold_vs_fixed"] = (
    attrib["intervention_rate_TrainRecall80"] -
    attrib["intervention_rate_Fixed0.50"]
)

display(attrib.round(5))
attrib.to_csv(
    RESULT_DIR / "e4_threshold_attribution.csv",
    index=False
)


# GroupDRO vs ERM probability-level effect

In [ ]:

prob_compare = model_metrics.pivot(
    index="heldout_group",
    columns="objective",
    values=["roc_auc", "pr_auc", "brier", "nll"]
)

rows = []
for heldout in groups:
    row = {"heldout_group": heldout}
    for metric in ["roc_auc", "pr_auc", "brier", "nll"]:
        erm = prob_compare.loc[heldout, (metric, "ERM")]
        dro = prob_compare.loc[heldout, (metric, "GroupDRO")]
        row[f"{metric}_ERM"] = erm
        row[f"{metric}_GroupDRO"] = dro
        if metric in ["roc_auc", "pr_auc"]:
            row[f"{metric}_DRO_improvement"] = dro - erm
        else:
            row[f"{metric}_DRO_improvement"] = erm - dro
    rows.append(row)

prob_effect = pd.DataFrame(rows)
display(prob_effect.round(6))
prob_effect.to_csv(
    RESULT_DIR / "e4_groupdro_vs_erm_probability_effect.csv",
    index=False
)


# Figures

In [ ]:

# Deployable recall by held-out group
fig_data = deployable.copy()
fig_data["condition"] = fig_data["objective"] + " + " + fig_data["policy"]

pivot = fig_data.pivot(
    index="heldout_group",
    columns="condition",
    values="recall"
)
ax = pivot.plot(kind="bar", figsize=(10, 5))
ax.axhline(TARGET_RECALL, linestyle="--", linewidth=1)
ax.set_ylabel("Recall")
ax.set_xlabel("Held-out Student_Type")
ax.set_title("E4 — Recall under unseen-population shift")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e4_recall_by_group_policy.png",
    dpi=200, bbox_inches="tight"
)
plt.show()

# Intervention burden
pivot2 = fig_data.pivot(
    index="heldout_group",
    columns="condition",
    values="intervention_rate"
)
ax = pivot2.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Intervention rate")
ax.set_xlabel("Held-out Student_Type")
ax.set_title("E4 — Intervention burden by policy")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e4_intervention_rate.png",
    dpi=200, bbox_inches="tight"
)
plt.show()


In [ ]:

# Worst-group recall vs weighted specificity
fig, ax = plt.subplots(figsize=(7, 5))
for _, row in aggregate.iterrows():
    label = f"{row['objective']} + {row['policy']}"
    ax.scatter(
        row["weighted_specificity"],
        row["worst_group_recall"],
        s=60,
    )
    ax.annotate(
        label,
        (row["weighted_specificity"], row["worst_group_recall"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

ax.axhline(TARGET_RECALL, linestyle="--", linewidth=1)
ax.set_xlabel("Weighted specificity")
ax.set_ylabel("Worst-group recall")
ax.set_title("E4 — Robust sensitivity vs specificity trade-off")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e4_worst_recall_vs_specificity.png",
    dpi=200, bbox_inches="tight"
)
plt.show()



# Interpretation Logic

## Evidence for a learning-objective problem

GroupDRO would need to improve one or more of:
- worst-group recall at the same threshold;
- worst-group balanced accuracy;
- PR-AUC / ROC-AUC on held-out populations;
- Brier/NLL;
- recall gap;

without an excessive intervention burden or specificity loss.

## Evidence for a decision-policy problem

If:
- ERM and GroupDRO have almost identical AUC/Brier/NLL,
- but `TrainRecall80` substantially improves worst-group recall,

then the dominant problem is not representation learning. It is **threshold-policy mismatch under differing population base rates / score distributions**.

## Meaning of the oracle

If the oracle threshold greatly exceeds deployable training-threshold performance, there remains target-population information that cannot be recovered from source populations alone.

If training-selected and oracle thresholds perform similarly, threshold adaptation is largely learnable without target labels.

## Important fairness language

Differences in subgroup performance are reported as **population heterogeneity / worst-group reliability**, not as a formal fairness violation. Formal fairness claims would require normative group definitions and a deployment context.


In [ ]:

manifest = {
    "experiment": "E4 ERM vs GroupDRO + Threshold Adaptation",
    "dataset_rows_known_group": int(len(e4_df)),
    "excluded_missing_group": int(df[GROUP].isna().sum()),
    "features": FEATURES,
    "group_variable": GROUP,
    "objectives": OBJECTIVES,
    "deployment_policies": ["Fixed0.50", "TrainRecall80"],
    "diagnostic_oracle": "HeldoutOracleRecall80 (non-deployable)",
    "target_recall": TARGET_RECALL,
    "inner_threshold_cv_folds": INNER_SPLITS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "l2": L2,
    "dro_eta": DRO_ETA,
    "seed": SEED,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}

with open(RESULT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved outputs:")
for p in sorted(RESULT_DIR.glob("*")):
    print("-", p)


# ========== E5 — Conformal Prediction + Selective Abstention ==========

The following cells are preserved from the validated standalone experiment notebook.


# E5 — Conformal Prediction + Selective Abstention under LOSO Population Shift

## Scientific objective

E4 showed that discrimination transports reasonably well, while a decision threshold learned on source populations does not transfer uniformly to an unseen `Student_Type`.

E5 asks:

> **Can conformal prediction expose uncertainty and support selective abstention when a decision policy is transported to an unseen population?**

## Primary methods

1. **Marginal split conformal**
   - one nonconformity threshold \(q\) for both classes.

2. **Mondrian / class-conditional conformal**
   - separate thresholds \(q_0\) and \(q_1\) for low- and high-stress labels.

The base predictor is the same **group-blind ERM Logistic Regression** retained from E1–E4.

## LOSO protocol

For each held-out `Student_Type` and each repeat:

- the two source populations are divided into:
  - **proper training set**: 60%
  - **conformal calibration set**: 20%
  - **source check set**: 20%
- the model is fitted only on proper training data;
- conformal thresholds are estimated only from the calibration split;
- the same fitted model and thresholds are evaluated on:
  - the source check set (**in-domain reference**),
  - the held-out population (**population-shift target**).

This allows a direct test of whether nominal conformal coverage transports from source to unseen population.

## Nominal coverage levels

\[
1-\alpha \in \{0.90,\;0.95\}
\]

## Selective abstention rule

- singleton prediction set, e.g. `{High}` or `{Low}` → **make a decision**;
- ambiguous set `{Low, High}` → **abstain**;
- empty set → **abstain**.

The conformal prediction set is therefore also used as an uncertainty-aware decision policy.

## Metrics

### Set-valued reliability
- marginal coverage,
- low-stress coverage,
- high-stress coverage,
- coverage gap,
- average set size,
- singleton rate,
- ambiguity rate,
- empty-set rate.

### Selective prediction
- decision coverage,
- selective accuracy,
- selective risk,
- actionable high-stress recall,
- high-stress abstention rate,
- false-negative singleton rate.

> Split-conformal finite-sample coverage requires exchangeability between calibration and test observations. LOSO deliberately violates that assumption at the population level, so target-domain undercoverage is a scientific result rather than a software failure.


In [ ]:

from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
TARGET = "Stress_Level"
GROUP = "Student_Type"

N_REPEATS = 5
ALPHAS = [0.10, 0.05]  # nominal 90%, 95%
TRAIN_FRAC = 0.60
CAL_FRAC = 0.20
CHECK_FRAC = 0.20

DATA_CANDIDATES = [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found. Edit DATA_PATH.")

RESULT_DIR = Path("../outputs/E5")
RESULT_DIR.mkdir(exist_ok=True)

print("Python :", platform.python_version())
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("Dataset:", DATA_PATH.resolve())
print("Repeats:", N_REPEATS)
print("Nominal coverage:", [1-a for a in ALPHAS])


In [ ]:

df = pd.read_csv(DATA_PATH)

NUMERIC_FEATURES = [
    "Sleep_Hours",
    "Study_Hours",
    "Social_Media_Hours",
    "Attendance",
    "Exam_Pressure",
    "Family_Support",
]
CATEGORICAL_FEATURES = ["Month"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

e5_df = (
    df[df[GROUP].notna()]
    .copy()
    .reset_index(drop=False)
    .rename(columns={"index": "original_row_id"})
)
groups = sorted(e5_df[GROUP].unique().tolist())

print("Full dataset:", df.shape)
print("E5 rows:", len(e5_df))
print("Excluded missing Student_Type:", int(df[GROUP].isna().sum()))
print("Groups:", groups)


# Base predictor

In [ ]:

def make_lr_pipeline():
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CATEGORICAL_FEATURES),
    ], sparse_threshold=0)

    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=SEED,
        )),
    ])



# Split-conformal construction

For a binary probabilistic classifier with \(p_1=P(Y=1|X)\) and \(p_0=1-p_1\), the nonconformity score for the observed label is:

\[
s_i = 1-p_{y_i}(x_i)
\]

For calibration scores \(s_1,\ldots,s_n\), the split-conformal threshold is the finite-sample order statistic:

\[
q_\alpha = s_{(k)}, \qquad
k=\left\lceil(n+1)(1-\alpha)\right\rceil
\]

with \(k\) capped at \(n\).

A candidate class \(c\) is included when:

\[
1-p_c(x)\le q_\alpha
\]

For Mondrian conformal, the same calculation is performed separately within each true calibration class.


In [ ]:

def true_label_scores(y_true, p1):
    y = np.asarray(y_true).astype(int)
    p1 = np.asarray(p1, dtype=float)
    p_true = np.where(y == 1, p1, 1 - p1)
    return 1 - p_true

def conformal_qhat(scores, alpha):
    scores = np.sort(np.asarray(scores, dtype=float))
    n = len(scores)
    if n == 0:
        return np.nan
    k = int(np.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(scores[k - 1])

def fit_conformal_thresholds(y_cal, p_cal, alpha, method):
    y = np.asarray(y_cal).astype(int)
    scores = true_label_scores(y, p_cal)

    if method == "marginal":
        q = conformal_qhat(scores, alpha)
        return {0: q, 1: q}

    if method == "mondrian":
        return {
            0: conformal_qhat(scores[y == 0], alpha),
            1: conformal_qhat(scores[y == 1], alpha),
        }

    raise ValueError("method must be 'marginal' or 'mondrian'")

def conformal_sets(p1, q_by_class):
    p1 = np.asarray(p1, dtype=float)
    p0 = 1 - p1

    include0 = (1 - p0) <= q_by_class[0]  # equivalently p1 <= q0
    include1 = (1 - p1) <= q_by_class[1]

    return include0.astype(int), include1.astype(int)

METHODS = ["marginal", "mondrian"]


# Evaluation metrics

In [ ]:

def set_metrics(y_true, include0, include1):
    y = np.asarray(y_true).astype(int)
    i0 = np.asarray(include0).astype(bool)
    i1 = np.asarray(include1).astype(bool)

    set_size = i0.astype(int) + i1.astype(int)
    covered = np.where(y == 0, i0, i1)

    mask0 = y == 0
    mask1 = y == 1

    cov0 = covered[mask0].mean() if mask0.any() else np.nan
    cov1 = covered[mask1].mean() if mask1.any() else np.nan

    singleton = set_size == 1
    ambiguous = set_size == 2
    empty = set_size == 0

    # singleton label
    singleton_pred = np.full(len(y), -1, dtype=int)
    singleton_pred[singleton & i0] = 0
    singleton_pred[singleton & i1] = 1

    decided = singleton
    if decided.any():
        selective_accuracy = (
            singleton_pred[decided] == y[decided]
        ).mean()
        selective_risk = 1 - selective_accuracy
    else:
        selective_accuracy = np.nan
        selective_risk = np.nan

    true_pos = mask1
    actionable_high_recall = (
        ((singleton_pred == 1) & true_pos).sum() / true_pos.sum()
        if true_pos.sum() else np.nan
    )
    high_abstention = (
        ((ambiguous | empty) & true_pos).sum() / true_pos.sum()
        if true_pos.sum() else np.nan
    )
    false_negative_singleton_rate = (
        ((singleton_pred == 0) & true_pos).sum() / true_pos.sum()
        if true_pos.sum() else np.nan
    )

    return {
        "marginal_coverage": covered.mean(),
        "coverage_low": cov0,
        "coverage_high": cov1,
        "coverage_gap_abs": abs(cov0 - cov1),
        "avg_set_size": set_size.mean(),
        "singleton_rate": singleton.mean(),
        "ambiguity_rate": ambiguous.mean(),
        "empty_rate": empty.mean(),
        "decision_coverage": decided.mean(),
        "selective_accuracy": selective_accuracy,
        "selective_risk": selective_risk,
        "actionable_high_stress_recall": actionable_high_recall,
        "high_stress_abstention_rate": high_abstention,
        "false_negative_singleton_rate": false_negative_singleton_rate,
    }

SET_METRICS = [
    "marginal_coverage", "coverage_low", "coverage_high",
    "coverage_gap_abs", "avg_set_size", "singleton_rate",
    "ambiguity_rate", "empty_rate", "decision_coverage",
    "selective_accuracy", "selective_risk",
    "actionable_high_stress_recall",
    "high_stress_abstention_rate",
    "false_negative_singleton_rate",
]



# Source split helper

The split is stratified jointly by `Student_Type × target` to preserve both source domains and both classes across proper-train, calibration, and source-check partitions.


In [ ]:

def joint_strata(frame):
    return (
        frame[GROUP].astype(str)
        + "__Y" + frame[TARGET].astype(int).astype(str)
    )

def source_three_way_split(source_df, seed):
    # First: 60% proper train, 40% temporary.
    train_df, temp_df = train_test_split(
        source_df,
        test_size=(CAL_FRAC + CHECK_FRAC),
        random_state=seed,
        stratify=joint_strata(source_df),
    )

    # Second: split temporary equally into calibration and source-check.
    relative_check = CHECK_FRAC / (CAL_FRAC + CHECK_FRAC)
    cal_df, check_df = train_test_split(
        temp_df,
        test_size=relative_check,
        random_state=seed + 1000,
        stratify=joint_strata(temp_df),
    )

    return (
        train_df.copy(),
        cal_df.copy(),
        check_df.copy(),
    )


# E5 LOSO experiment

In [ ]:

metric_rows = []
threshold_rows = []
prediction_rows = []
base_discrimination_rows = []

for heldout in groups:
    source_df = e5_df[e5_df[GROUP] != heldout].copy()
    target_df = e5_df[e5_df[GROUP] == heldout].copy()

    for repeat in range(1, N_REPEATS + 1):
        seed = SEED + repeat * 101

        proper_train, cal_df, source_check = source_three_way_split(
            source_df, seed
        )

        model = make_lr_pipeline()
        model.fit(
            proper_train[FEATURES],
            proper_train[TARGET].astype(int),
        )

        p_cal = model.predict_proba(cal_df[FEATURES])[:, 1]
        p_source = model.predict_proba(source_check[FEATURES])[:, 1]
        p_target = model.predict_proba(target_df[FEATURES])[:, 1]

        # Context: discrimination only.
        for domain_name, part, prob in [
            ("source_check", source_check, p_source),
            ("target_loso", target_df, p_target),
        ]:
            y_dom = part[TARGET].astype(int).to_numpy()
            base_discrimination_rows.append({
                "heldout_group": heldout,
                "repeat": repeat,
                "domain": domain_name,
                "n": len(part),
                "prevalence": y_dom.mean(),
                "roc_auc": roc_auc_score(y_dom, prob),
                "pr_auc": average_precision_score(y_dom, prob),
            })

        for alpha in ALPHAS:
            nominal = 1 - alpha

            for method in METHODS:
                q_by_class = fit_conformal_thresholds(
                    cal_df[TARGET].astype(int).to_numpy(),
                    p_cal,
                    alpha=alpha,
                    method=method,
                )

                threshold_rows.append({
                    "heldout_group": heldout,
                    "repeat": repeat,
                    "method": method,
                    "alpha": alpha,
                    "nominal_coverage": nominal,
                    "n_cal": len(cal_df),
                    "n_cal_low": int((cal_df[TARGET] == 0).sum()),
                    "n_cal_high": int((cal_df[TARGET] == 1).sum()),
                    "q_low": q_by_class[0],
                    "q_high": q_by_class[1],
                })

                for domain_name, part, prob in [
                    ("source_check", source_check, p_source),
                    ("target_loso", target_df, p_target),
                ]:
                    y_dom = part[TARGET].astype(int).to_numpy()
                    i0, i1 = conformal_sets(prob, q_by_class)

                    metrics = set_metrics(y_dom, i0, i1)
                    metric_rows.append({
                        "heldout_group": heldout,
                        "repeat": repeat,
                        "domain": domain_name,
                        "method": method,
                        "alpha": alpha,
                        "nominal_coverage": nominal,
                        "n": len(part),
                        "prevalence": y_dom.mean(),
                        **metrics,
                    })

                    # Row-level target predictions are useful for later E6.
                    if domain_name == "target_loso":
                        for rid, yt, pr, a0, a1 in zip(
                            part["original_row_id"].to_numpy(),
                            y_dom,
                            prob,
                            i0,
                            i1,
                        ):
                            prediction_rows.append({
                                "original_row_id": int(rid),
                                "heldout_group": heldout,
                                "repeat": repeat,
                                "method": method,
                                "alpha": alpha,
                                "y_true": int(yt),
                                "prob_high": float(pr),
                                "include_low": int(a0),
                                "include_high": int(a1),
                                "set_size": int(a0) + int(a1),
                            })

metrics = pd.DataFrame(metric_rows)
thresholds = pd.DataFrame(threshold_rows)
predictions = pd.DataFrame(prediction_rows)
base_disc = pd.DataFrame(base_discrimination_rows)

metrics.to_csv(
    RESULT_DIR / "e5_conformal_metrics_by_repeat.csv",
    index=False
)
thresholds.to_csv(
    RESULT_DIR / "e5_conformal_thresholds.csv",
    index=False
)
predictions.to_csv(
    RESULT_DIR / "e5_target_conformal_predictions.csv",
    index=False
)
base_disc.to_csv(
    RESULT_DIR / "e5_base_discrimination.csv",
    index=False
)

display(metrics.head())


# Mean LOSO results across repeats

In [ ]:

summary = (
    metrics
    .groupby(
        ["heldout_group", "domain", "method", "alpha", "nominal_coverage"]
    )[SET_METRICS]
    .agg(["mean", "std"])
)
display(summary)
summary.to_csv(
    RESULT_DIR / "e5_summary_mean_sd.csv"
)



# Coverage transportability

Define target coverage deterioration as:

\[
\Delta Coverage =
Coverage_{source-check} - Coverage_{target-LOSO}
\]

Positive values mean the conformal set covers the true label less often after population shift.


In [ ]:

mean_metrics = (
    metrics
    .groupby(
        ["heldout_group", "domain", "method", "alpha", "nominal_coverage"],
        as_index=False
    )
    .agg(
        n=("n", "mean"),
        prevalence=("prevalence", "mean"),
        **{m: (m, "mean") for m in SET_METRICS},
    )
)

source_mean = mean_metrics[
    mean_metrics["domain"] == "source_check"
].drop(columns="domain")

target_mean = mean_metrics[
    mean_metrics["domain"] == "target_loso"
].drop(columns="domain")

transport = source_mean.merge(
    target_mean,
    on=["heldout_group", "method", "alpha", "nominal_coverage"],
    suffixes=("_source", "_target"),
)

for metric in [
    "marginal_coverage", "coverage_low", "coverage_high",
    "selective_accuracy", "decision_coverage",
]:
    transport[f"{metric}_transport_change"] = (
        transport[f"{metric}_source"] -
        transport[f"{metric}_target"]
    )

transport["target_undercoverage_vs_nominal"] = (
    transport["nominal_coverage"] -
    transport["marginal_coverage_target"]
)
transport["target_high_undercoverage_vs_nominal"] = (
    transport["nominal_coverage"] -
    transport["coverage_high_target"]
)

display(
    transport[
        [
            "heldout_group", "method", "nominal_coverage",
            "marginal_coverage_source", "marginal_coverage_target",
            "target_undercoverage_vs_nominal",
            "coverage_high_source", "coverage_high_target",
            "target_high_undercoverage_vs_nominal",
            "decision_coverage_target", "selective_accuracy_target",
            "actionable_high_stress_recall_target",
            "high_stress_abstention_rate_target",
        ]
    ].round(5)
)

transport.to_csv(
    RESULT_DIR / "e5_coverage_transportability.csv",
    index=False
)


# Weighted and worst-group target summary

In [ ]:

target_only = mean_metrics[
    mean_metrics["domain"] == "target_loso"
].copy()

weighted_rows = []
for (method, alpha), part in target_only.groupby(["method", "alpha"]):
    w = part["n"] / part["n"].sum()
    nominal = float(part["nominal_coverage"].iloc[0])

    weighted_rows.append({
        "method": method,
        "alpha": alpha,
        "nominal_coverage": nominal,
        "weighted_marginal_coverage": np.sum(
            w * part["marginal_coverage"]
        ),
        "worst_group_marginal_coverage": part["marginal_coverage"].min(),
        "weighted_high_coverage": np.sum(
            w * part["coverage_high"]
        ),
        "worst_group_high_coverage": part["coverage_high"].min(),
        "weighted_avg_set_size": np.sum(
            w * part["avg_set_size"]
        ),
        "weighted_decision_coverage": np.sum(
            w * part["decision_coverage"]
        ),
        "worst_group_decision_coverage": part["decision_coverage"].min(),
        "weighted_selective_accuracy": np.sum(
            w * part["selective_accuracy"]
        ),
        "worst_group_selective_accuracy": part["selective_accuracy"].min(),
        "weighted_actionable_high_recall": np.sum(
            w * part["actionable_high_stress_recall"]
        ),
        "worst_group_actionable_high_recall": (
            part["actionable_high_stress_recall"].min()
        ),
        "weighted_high_abstention": np.sum(
            w * part["high_stress_abstention_rate"]
        ),
    })

weighted_summary = pd.DataFrame(weighted_rows)
display(weighted_summary.round(5))
weighted_summary.to_csv(
    RESULT_DIR / "e5_target_weighted_worst_summary.csv",
    index=False
)


# Selective-abstention detail by unseen population

In [ ]:

selective_table = target_only[
    [
        "heldout_group", "method", "nominal_coverage",
        "decision_coverage", "selective_accuracy", "selective_risk",
        "actionable_high_stress_recall",
        "high_stress_abstention_rate",
        "false_negative_singleton_rate",
        "avg_set_size", "ambiguity_rate", "empty_rate",
    ]
].sort_values(
    ["nominal_coverage", "heldout_group", "method"]
)

display(selective_table.round(5))
selective_table.to_csv(
    RESULT_DIR / "e5_selective_abstention_target.csv",
    index=False
)


# Figures

In [ ]:

# 1. Target marginal coverage vs nominal
for nominal in sorted(target_only["nominal_coverage"].unique()):
    part = target_only[target_only["nominal_coverage"] == nominal]
    pivot = part.pivot(
        index="heldout_group",
        columns="method",
        values="marginal_coverage",
    )
    ax = pivot.plot(kind="bar", figsize=(8, 4))
    ax.axhline(nominal, linestyle="--", linewidth=1)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Marginal coverage")
    ax.set_xlabel("Held-out Student_Type")
    ax.set_title(
        f"E5 — Target coverage under LOSO (nominal={nominal:.0%})"
    )
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(
        RESULT_DIR / f"fig_e5_target_coverage_{int(nominal*100)}.png",
        dpi=200, bbox_inches="tight"
    )
    plt.show()

# 2. High-stress class coverage
for nominal in sorted(target_only["nominal_coverage"].unique()):
    part = target_only[target_only["nominal_coverage"] == nominal]
    pivot = part.pivot(
        index="heldout_group",
        columns="method",
        values="coverage_high",
    )
    ax = pivot.plot(kind="bar", figsize=(8, 4))
    ax.axhline(nominal, linestyle="--", linewidth=1)
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Coverage for true high-stress cases")
    ax.set_xlabel("Held-out Student_Type")
    ax.set_title(
        f"E5 — High-stress coverage (nominal={nominal:.0%})"
    )
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(
        RESULT_DIR / f"fig_e5_high_coverage_{int(nominal*100)}.png",
        dpi=200, bbox_inches="tight"
    )
    plt.show()


In [ ]:

# 3. Decision coverage vs selective accuracy
fig, ax = plt.subplots(figsize=(8, 5))

for _, row in target_only.iterrows():
    label = (
        f"{row['heldout_group']} | "
        f"{row['method']} | "
        f"{row['nominal_coverage']:.0%}"
    )
    ax.scatter(
        row["decision_coverage"],
        row["selective_accuracy"],
        s=55,
    )
    ax.annotate(
        label,
        (row["decision_coverage"], row["selective_accuracy"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=7,
    )

ax.set_xlabel("Decision coverage (singleton rate)")
ax.set_ylabel("Selective accuracy")
ax.set_title("E5 — Accuracy–abstention trade-off under LOSO")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e5_selective_accuracy_vs_coverage.png",
    dpi=200, bbox_inches="tight"
)
plt.show()



# Optional target-label oracle diagnostic

The following diagnostic asks:

> If a small labeled sample from the target population were available solely for conformal recalibration, how much of the target undercoverage could be recovered?

This is **not** a zero-shot LOSO method and is never mixed with the primary results.

For each repeat:
- keep the source-trained classifier unchanged;
- split the held-out population 50/50;
- use one half only to recompute conformal thresholds;
- evaluate those thresholds on the other half.

This isolates **conformal-threshold transportability** from model transportability.


In [ ]:

oracle_rows = []

for heldout in groups:
    source_df = e5_df[e5_df[GROUP] != heldout].copy()
    target_df = e5_df[e5_df[GROUP] == heldout].copy()

    for repeat in range(1, N_REPEATS + 1):
        seed = SEED + repeat * 101

        proper_train, cal_df, source_check = source_three_way_split(
            source_df, seed
        )

        model = make_lr_pipeline()
        model.fit(
            proper_train[FEATURES],
            proper_train[TARGET].astype(int),
        )

        # Target split used only for the oracle diagnostic.
        target_cal, target_eval = train_test_split(
            target_df,
            test_size=0.50,
            random_state=seed + 5000,
            stratify=target_df[TARGET],
        )

        p_tcal = model.predict_proba(target_cal[FEATURES])[:, 1]
        p_teval = model.predict_proba(target_eval[FEATURES])[:, 1]

        for alpha in ALPHAS:
            for method in METHODS:
                q_oracle = fit_conformal_thresholds(
                    target_cal[TARGET].astype(int).to_numpy(),
                    p_tcal,
                    alpha,
                    method,
                )
                i0, i1 = conformal_sets(p_teval, q_oracle)

                oracle_rows.append({
                    "heldout_group": heldout,
                    "repeat": repeat,
                    "method": method,
                    "alpha": alpha,
                    "nominal_coverage": 1-alpha,
                    "diagnostic": "target_label_oracle_recalibration",
                    "n_target_cal": len(target_cal),
                    "n_target_eval": len(target_eval),
                    **set_metrics(
                        target_eval[TARGET].astype(int).to_numpy(),
                        i0, i1
                    ),
                })

oracle = pd.DataFrame(oracle_rows)
oracle_summary = (
    oracle
    .groupby(
        ["heldout_group", "method", "alpha", "nominal_coverage"]
    )[SET_METRICS]
    .agg(["mean", "std"])
)

display(oracle_summary)
oracle.to_csv(
    RESULT_DIR / "e5_oracle_target_recalibration_by_repeat.csv",
    index=False
)
oracle_summary.to_csv(
    RESULT_DIR / "e5_oracle_target_recalibration_summary.csv"
)



# Interpretation logic

## Result pattern A — source coverage near nominal, target coverage below nominal

This is direct evidence that **split-conformal coverage does not transport under population shift**, consistent with the failure of exchangeability.

## Result pattern B — Mondrian improves high-stress coverage

Then class imbalance / asymmetric score behavior is an important part of the uncertainty problem.

However, class-conditional coverage on source data still does not guarantee target-population coverage under shift.

## Result pattern C — singleton decisions are highly accurate but sparse

This supports **selective abstention**:
the system becomes more reliable by refusing ambiguous cases, at the cost of intervention/decision coverage.

## Result pattern D — actionable high-stress recall becomes low

Then conformal abstention is safe in a statistical sense but may be operationally inadequate for screening. This trade-off must be made explicit.

## Oracle interpretation

If target-label recalibration restores nominal coverage while the source-calibrated conformal method undercovers, the dominant failure is **conformal threshold transportability** rather than the ranking model itself.

## Claims to avoid

Do not claim:
- guaranteed target coverage under LOSO,
- clinical safety,
- causal inference,
- formal fairness guarantees.

The correct claim is empirical:
**uncertainty guarantees derived under source exchangeability can degrade under unseen population shift.**


In [ ]:

manifest = {
    "experiment": "E5 Conformal Prediction + Selective Abstention under LOSO",
    "dataset_rows_known_group": int(len(e5_df)),
    "excluded_missing_group": int(df[GROUP].isna().sum()),
    "base_model": "group-blind ERM Logistic Regression",
    "features": FEATURES,
    "groups": groups,
    "methods": METHODS,
    "alphas": ALPHAS,
    "nominal_coverages": [1-a for a in ALPHAS],
    "n_repeats": N_REPEATS,
    "source_split": {
        "proper_train": TRAIN_FRAC,
        "calibration": CAL_FRAC,
        "source_check": CHECK_FRAC,
    },
    "selective_rule": "decide only for singleton conformal prediction sets",
    "oracle": "target-label conformal recalibration diagnostic; non-deployable",
    "seed": SEED,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}

with open(
    RESULT_DIR / "run_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2)

print("Saved outputs:")
for p in sorted(RESULT_DIR.glob("*")):
    print("-", p)


# ========== E6 — Capacity-Constrained Intervention Policy ==========

The following cells are preserved from the validated standalone experiment notebook.


# E6 — Capacity-Constrained Intervention Policy under LOSO Population Shift

## Scientific objective

E4 showed that a source-derived decision threshold does not transfer uniformly across unseen student populations.  
E5 showed that class-conditional conformal prediction can reduce hidden high-stress undercoverage, but uncertainty sets also create abstention burden.

E6 translates those findings into an operational question:

> **When an institution can intervene for only a limited fraction of students, which policy captures the most high-stress cases while controlling intervention burden?**

## LOSO design

For each held-out `Student_Type` and each repeat:

1. the other two populations are source data;
2. source data are split 80/20 into:
   - proper training,
   - source calibration;
3. group-blind Logistic Regression is fitted on proper training only;
4. the calibration split is used to learn:
   - a Recall≥0.80 threshold,
   - Mondrian conformal thresholds at nominal 90% coverage;
5. the held-out population remains label-unseen during policy construction;
6. labels are used only for final evaluation.

## Capacity levels

\[
K \in \{10\%, 15\%, 20\%, 25\%\}
\]

## Capacity-constrained policies

### 1. RiskTopK
Select the highest predicted risks until capacity \(K\) is full.

### 2. ConformalHighOnly
Only singleton `{High}` cases are eligible.  
If fewer singleton-high cases exist than capacity, capacity is intentionally left unused.

### 3. ConformalTriageTopK
Priority:
1. singleton `{High}`;
2. ambiguous `{Low, High}`, ranked by predicted risk;
3. singleton `{Low}`, only if capacity still remains.

This represents uncertainty-aware triage rather than pure score ranking.

## Non-capacity benchmark

### SourceRecall80
Apply the source-calibration threshold satisfying:

\[
Recall \ge 0.80
\]

This policy is not forced to obey \(K\). It estimates how much intervention capacity would actually be required to pursue a sensitivity target learned from source populations.

## Primary operational metrics

- Precision@K
- Recall@K
- Lift@K
- high-stress cases captured
- number needed to review (NNR)
- intervention rate
- capacity utilization
- false-positive interventions
- missed high-stress cases
- uncertainty share among selected cases
- worst-group Recall@K across held-out populations


In [ ]:

from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve

SEED = 42
TARGET = "Stress_Level"
GROUP = "Student_Type"

N_REPEATS = 5
SOURCE_CAL_FRAC = 0.20
TARGET_RECALL = 0.80
ALPHA = 0.10  # Mondrian 90% nominal coverage
CAPACITIES = [0.10, 0.15, 0.20, 0.25]

DATA_CANDIDATES = [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found. Edit DATA_PATH.")

RESULT_DIR = Path("../outputs/E6")
RESULT_DIR.mkdir(exist_ok=True)

print("Python :", platform.python_version())
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("Dataset:", DATA_PATH.resolve())
print("Repeats:", N_REPEATS)
print("Capacities:", CAPACITIES)


In [ ]:

df = pd.read_csv(DATA_PATH)

NUMERIC_FEATURES = [
    "Sleep_Hours",
    "Study_Hours",
    "Social_Media_Hours",
    "Attendance",
    "Exam_Pressure",
    "Family_Support",
]
CATEGORICAL_FEATURES = ["Month"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

e6_df = (
    df[df[GROUP].notna()]
    .copy()
    .reset_index(drop=False)
    .rename(columns={"index": "original_row_id"})
)
groups = sorted(e6_df[GROUP].unique().tolist())

print("Rows used:", len(e6_df))
print("Excluded missing Student_Type:", int(df[GROUP].isna().sum()))
print("Groups:", groups)


# Shared group-blind model

In [ ]:

def make_lr_pipeline():
    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CATEGORICAL_FEATURES),
    ], sparse_threshold=0)

    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(
            max_iter=2000,
            solver="lbfgs",
            random_state=SEED,
        )),
    ])


# Threshold and Mondrian conformal utilities

In [ ]:

def joint_strata(frame):
    return (
        frame[GROUP].astype(str)
        + "__Y" + frame[TARGET].astype(int).astype(str)
    )

def select_recall_constrained_threshold(y_true, prob, target_recall=0.80):
    y = np.asarray(y_true).astype(int)
    p = np.asarray(prob, dtype=float)
    precision, recall, thresholds = precision_recall_curve(y, p)

    if len(thresholds) == 0:
        return 0.5

    valid = recall[:-1] >= target_recall
    if not np.any(valid):
        return 0.0

    return float(np.max(thresholds[valid]))

def true_label_scores(y_true, p1):
    y = np.asarray(y_true).astype(int)
    p1 = np.asarray(p1, dtype=float)
    p_true = np.where(y == 1, p1, 1 - p1)
    return 1 - p_true

def conformal_qhat(scores, alpha):
    scores = np.sort(np.asarray(scores, dtype=float))
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(scores[k - 1])

def fit_mondrian_thresholds(y_cal, p_cal, alpha=ALPHA):
    y = np.asarray(y_cal).astype(int)
    scores = true_label_scores(y, p_cal)
    return {
        0: conformal_qhat(scores[y == 0], alpha),
        1: conformal_qhat(scores[y == 1], alpha),
    }

def mondrian_sets(p1, q):
    p1 = np.asarray(p1, dtype=float)
    p0 = 1 - p1
    include0 = (1 - p0) <= q[0]
    include1 = (1 - p1) <= q[1]
    set_size = include0.astype(int) + include1.astype(int)

    singleton_high = include1 & ~include0
    singleton_low = include0 & ~include1
    ambiguous = include0 & include1
    empty = ~include0 & ~include1

    return pd.DataFrame({
        "include_low": include0.astype(int),
        "include_high": include1.astype(int),
        "set_size": set_size,
        "singleton_high": singleton_high.astype(int),
        "singleton_low": singleton_low.astype(int),
        "ambiguous": ambiguous.astype(int),
        "empty": empty.astype(int),
    })


# Operational policy utilities

In [ ]:

def evaluate_selection(y_true, selected, capacity_n=None, uncertain=None):
    y = np.asarray(y_true).astype(int)
    sel = np.asarray(selected).astype(bool)
    n = len(y)

    selected_n = int(sel.sum())
    positives = int(y.sum())
    captured = int((y[sel] == 1).sum()) if selected_n else 0
    fp = selected_n - captured
    missed = positives - captured

    precision = captured / selected_n if selected_n else np.nan
    recall = captured / positives if positives else np.nan
    prevalence = positives / n if n else np.nan
    lift = precision / prevalence if selected_n and prevalence > 0 else np.nan
    nnr = 1 / precision if precision and precision > 0 else np.nan

    if capacity_n is None:
        capacity_utilization = np.nan
    else:
        capacity_utilization = selected_n / capacity_n if capacity_n > 0 else np.nan

    uncertainty_share = np.nan
    if uncertain is not None and selected_n:
        uncertainty_share = np.asarray(uncertain).astype(bool)[sel].mean()

    return {
        "n": n,
        "prevalence": prevalence,
        "selected_n": selected_n,
        "intervention_rate": selected_n / n,
        "capacity_n": capacity_n,
        "capacity_utilization": capacity_utilization,
        "high_stress_captured": captured,
        "precision_at_k": precision,
        "recall_at_k": recall,
        "lift_at_k": lift,
        "number_needed_to_review": nnr,
        "false_positive_interventions": fp,
        "missed_high_stress": missed,
        "uncertainty_share_selected": uncertainty_share,
    }

def select_risk_topk(prob, capacity_n):
    order = np.argsort(-np.asarray(prob))
    selected = np.zeros(len(prob), dtype=bool)
    selected[order[:capacity_n]] = True
    return selected

def select_conformal_high_only(conf, prob, capacity_n):
    eligible = np.where(conf["singleton_high"].to_numpy().astype(bool))[0]
    eligible = eligible[np.argsort(-np.asarray(prob)[eligible])]

    selected = np.zeros(len(prob), dtype=bool)
    selected[eligible[:capacity_n]] = True
    return selected

def select_conformal_triage(conf, prob, capacity_n):
    p = np.asarray(prob)

    tiers = np.full(len(p), -1, dtype=int)
    tiers[conf["singleton_low"].to_numpy().astype(bool)] = 0
    tiers[conf["ambiguous"].to_numpy().astype(bool)] = 1
    tiers[conf["singleton_high"].to_numpy().astype(bool)] = 2
    tiers[conf["empty"].to_numpy().astype(bool)] = -1

    # descending tier, then descending probability
    order = np.lexsort((-p, -tiers))

    selected = np.zeros(len(p), dtype=bool)
    selected[order[:capacity_n]] = True
    return selected


# E6 LOSO capacity experiment

In [ ]:

capacity_rows = []
benchmark_rows = []
threshold_rows = []
row_predictions = []

for heldout in groups:
    source_df = e6_df[e6_df[GROUP] != heldout].copy()
    target_df = e6_df[e6_df[GROUP] == heldout].copy()

    for repeat in range(1, N_REPEATS + 1):
        seed = SEED + repeat * 131

        proper_train, cal_df = train_test_split(
            source_df,
            test_size=SOURCE_CAL_FRAC,
            random_state=seed,
            stratify=joint_strata(source_df),
        )

        model = make_lr_pipeline()
        model.fit(
            proper_train[FEATURES],
            proper_train[TARGET].astype(int),
        )

        p_cal = model.predict_proba(cal_df[FEATURES])[:, 1]
        p_target = model.predict_proba(target_df[FEATURES])[:, 1]
        y_target = target_df[TARGET].astype(int).to_numpy()

        recall80_threshold = select_recall_constrained_threshold(
            cal_df[TARGET].astype(int).to_numpy(),
            p_cal,
            TARGET_RECALL,
        )

        q = fit_mondrian_thresholds(
            cal_df[TARGET].astype(int).to_numpy(),
            p_cal,
            alpha=ALPHA,
        )

        conf = mondrian_sets(p_target, q)
        uncertain = (
            conf["ambiguous"].to_numpy().astype(bool)
            | conf["empty"].to_numpy().astype(bool)
        )

        threshold_rows.append({
            "heldout_group": heldout,
            "repeat": repeat,
            "recall80_threshold": recall80_threshold,
            "mondrian_q_low": q[0],
            "mondrian_q_high": q[1],
            "calibration_n": len(cal_df),
            "target_n": len(target_df),
        })

        # Capacity-free benchmark: source-calibrated recall target
        selected_threshold = p_target >= recall80_threshold
        benchmark_rows.append({
            "heldout_group": heldout,
            "repeat": repeat,
            "policy": "SourceRecall80",
            **evaluate_selection(
                y_target,
                selected_threshold,
                capacity_n=None,
                uncertain=uncertain,
            ),
        })

        # Capacity-free conformal high-only benchmark
        selected_singleton = (
            conf["singleton_high"].to_numpy().astype(bool)
        )
        benchmark_rows.append({
            "heldout_group": heldout,
            "repeat": repeat,
            "policy": "MondrianSingletonHigh90",
            **evaluate_selection(
                y_target,
                selected_singleton,
                capacity_n=None,
                uncertain=uncertain,
            ),
        })

        # Fixed capacities
        for k in CAPACITIES:
            capacity_n = max(1, int(np.ceil(k * len(target_df))))

            selections = {
                "RiskTopK": select_risk_topk(
                    p_target, capacity_n
                ),
                "ConformalHighOnly": select_conformal_high_only(
                    conf, p_target, capacity_n
                ),
                "ConformalTriageTopK": select_conformal_triage(
                    conf, p_target, capacity_n
                ),
            }

            for policy, selected in selections.items():
                capacity_rows.append({
                    "heldout_group": heldout,
                    "repeat": repeat,
                    "capacity": k,
                    "policy": policy,
                    **evaluate_selection(
                        y_target,
                        selected,
                        capacity_n=capacity_n,
                        uncertain=uncertain,
                    ),
                })

        # Row-level output for later audit/manuscript diagnostics
        for rid, yt, pr, sh, sl, amb, emp in zip(
            target_df["original_row_id"].to_numpy(),
            y_target,
            p_target,
            conf["singleton_high"].to_numpy(),
            conf["singleton_low"].to_numpy(),
            conf["ambiguous"].to_numpy(),
            conf["empty"].to_numpy(),
        ):
            row_predictions.append({
                "original_row_id": int(rid),
                "heldout_group": heldout,
                "repeat": repeat,
                "y_true": int(yt),
                "prob_high": float(pr),
                "singleton_high": int(sh),
                "singleton_low": int(sl),
                "ambiguous": int(amb),
                "empty": int(emp),
                "recall80_threshold": recall80_threshold,
            })

capacity_results = pd.DataFrame(capacity_rows)
benchmarks = pd.DataFrame(benchmark_rows)
thresholds = pd.DataFrame(threshold_rows)
predictions = pd.DataFrame(row_predictions)

capacity_results.to_csv(
    RESULT_DIR / "e6_capacity_results_by_repeat.csv",
    index=False
)
benchmarks.to_csv(
    RESULT_DIR / "e6_noncapacity_benchmarks_by_repeat.csv",
    index=False
)
thresholds.to_csv(
    RESULT_DIR / "e6_source_policy_thresholds.csv",
    index=False
)
predictions.to_csv(
    RESULT_DIR / "e6_target_row_predictions.csv",
    index=False
)

display(capacity_results.head())


# Mean capacity performance by unseen population

In [ ]:

CAP_METRICS = [
    "intervention_rate", "capacity_utilization",
    "high_stress_captured", "precision_at_k", "recall_at_k",
    "lift_at_k", "number_needed_to_review",
    "false_positive_interventions", "missed_high_stress",
    "uncertainty_share_selected",
]

capacity_summary = (
    capacity_results
    .groupby(
        ["heldout_group", "capacity", "policy"],
        as_index=False
    )[CAP_METRICS]
    .mean()
)

display(
    capacity_summary[
        [
            "heldout_group", "capacity", "policy",
            "precision_at_k", "recall_at_k", "lift_at_k",
            "number_needed_to_review", "capacity_utilization",
            "intervention_rate", "uncertainty_share_selected",
        ]
    ].round(5)
)

capacity_summary.to_csv(
    RESULT_DIR / "e6_capacity_summary_by_group.csv",
    index=False
)


# Weighted and worst-group capacity summary

In [ ]:

aggregate_rows = []

for (capacity, policy), part in capacity_summary.groupby(
    ["capacity", "policy"]
):
    # Target sizes are constant across repeats; recover from source results.
    sizes = (
        capacity_results[
            (capacity_results["capacity"] == capacity) &
            (capacity_results["policy"] == policy)
        ]
        .groupby("heldout_group")["n"]
        .first()
    )
    part2 = part.set_index("heldout_group").loc[sizes.index]
    w = sizes / sizes.sum()

    aggregate_rows.append({
        "capacity": capacity,
        "policy": policy,
        "weighted_precision_at_k": np.sum(
            w * part2["precision_at_k"]
        ),
        "weighted_recall_at_k": np.sum(
            w * part2["recall_at_k"]
        ),
        "worst_group_recall_at_k": part2["recall_at_k"].min(),
        "best_group_recall_at_k": part2["recall_at_k"].max(),
        "recall_gap": (
            part2["recall_at_k"].max()
            - part2["recall_at_k"].min()
        ),
        "weighted_lift_at_k": np.sum(
            w * part2["lift_at_k"]
        ),
        "weighted_nnr": np.sum(
            w * part2["number_needed_to_review"]
        ),
        "weighted_capacity_utilization": np.sum(
            w * part2["capacity_utilization"]
        ),
        "weighted_uncertainty_share_selected": np.nansum(
            w * part2["uncertainty_share_selected"].fillna(0)
        ),
    })

aggregate = pd.DataFrame(aggregate_rows)
display(aggregate.round(5))

aggregate.to_csv(
    RESULT_DIR / "e6_capacity_weighted_worst_summary.csv",
    index=False
)


# Non-capacity benchmark requirements

In [ ]:

benchmark_summary = (
    benchmarks
    .groupby(["heldout_group", "policy"], as_index=False)
    .agg(
        intervention_rate=("intervention_rate", "mean"),
        precision=("precision_at_k", "mean"),
        recall=("recall_at_k", "mean"),
        lift=("lift_at_k", "mean"),
        nnr=("number_needed_to_review", "mean"),
        high_stress_captured=("high_stress_captured", "mean"),
        missed_high_stress=("missed_high_stress", "mean"),
    )
)

display(benchmark_summary.round(5))
benchmark_summary.to_csv(
    RESULT_DIR / "e6_noncapacity_benchmark_summary.csv",
    index=False
)



# Capacity feasibility of the source Recall≥0.80 policy

For each held-out population, compare the intervention rate required by `SourceRecall80` with the fixed institutional capacity levels.

A policy is capacity-feasible when:

\[
InterventionRate \le K
\]


In [ ]:

feasibility_rows = []

source_recall80 = benchmark_summary[
    benchmark_summary["policy"] == "SourceRecall80"
]

for _, row in source_recall80.iterrows():
    for k in CAPACITIES:
        feasibility_rows.append({
            "heldout_group": row["heldout_group"],
            "capacity": k,
            "required_intervention_rate": row["intervention_rate"],
            "capacity_feasible": row["intervention_rate"] <= k,
            "capacity_shortfall": max(
                0.0, row["intervention_rate"] - k
            ),
            "achieved_recall": row["recall"],
        })

feasibility = pd.DataFrame(feasibility_rows)
display(feasibility)
feasibility.to_csv(
    RESULT_DIR / "e6_recall80_capacity_feasibility.csv",
    index=False
)


# Direct policy attribution

In [ ]:

# Compare uncertainty-aware triage and strict conformal eligibility against pure risk ranking.
risk = aggregate[aggregate["policy"] == "RiskTopK"].set_index("capacity")

attrib_rows = []
for policy in ["ConformalHighOnly", "ConformalTriageTopK"]:
    cur = aggregate[aggregate["policy"] == policy].set_index("capacity")

    for k in CAPACITIES:
        attrib_rows.append({
            "capacity": k,
            "policy": policy,
            "delta_weighted_recall_vs_risk": (
                cur.loc[k, "weighted_recall_at_k"]
                - risk.loc[k, "weighted_recall_at_k"]
            ),
            "delta_worst_group_recall_vs_risk": (
                cur.loc[k, "worst_group_recall_at_k"]
                - risk.loc[k, "worst_group_recall_at_k"]
            ),
            "delta_precision_vs_risk": (
                cur.loc[k, "weighted_precision_at_k"]
                - risk.loc[k, "weighted_precision_at_k"]
            ),
            "delta_lift_vs_risk": (
                cur.loc[k, "weighted_lift_at_k"]
                - risk.loc[k, "weighted_lift_at_k"]
            ),
            "capacity_utilization": cur.loc[
                k, "weighted_capacity_utilization"
            ],
        })

attribution = pd.DataFrame(attrib_rows)
display(attribution.round(5))
attribution.to_csv(
    RESULT_DIR / "e6_policy_attribution_vs_risk.csv",
    index=False
)


# Figures

In [ ]:

# Recall@K
fig, ax = plt.subplots(figsize=(8, 5))
for policy, part in aggregate.groupby("policy"):
    ax.plot(
        part["capacity"] * 100,
        part["weighted_recall_at_k"],
        marker="o",
        label=policy,
    )
ax.set_xlabel("Intervention capacity (%)")
ax.set_ylabel("Weighted Recall@K")
ax.set_title("E6 — High-stress capture under fixed capacity")
ax.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e6_recall_at_k.png",
    dpi=200, bbox_inches="tight"
)
plt.show()

# Precision@K
fig, ax = plt.subplots(figsize=(8, 5))
for policy, part in aggregate.groupby("policy"):
    ax.plot(
        part["capacity"] * 100,
        part["weighted_precision_at_k"],
        marker="o",
        label=policy,
    )
ax.set_xlabel("Intervention capacity (%)")
ax.set_ylabel("Weighted Precision@K")
ax.set_title("E6 — Precision under fixed capacity")
ax.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e6_precision_at_k.png",
    dpi=200, bbox_inches="tight"
)
plt.show()


In [ ]:

# Worst-group recall
fig, ax = plt.subplots(figsize=(8, 5))
for policy, part in aggregate.groupby("policy"):
    ax.plot(
        part["capacity"] * 100,
        part["worst_group_recall_at_k"],
        marker="o",
        label=policy,
    )
ax.set_xlabel("Intervention capacity (%)")
ax.set_ylabel("Worst-group Recall@K")
ax.set_title("E6 — Worst-population high-stress capture")
ax.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e6_worst_group_recall_at_k.png",
    dpi=200, bbox_inches="tight"
)
plt.show()

# Recall vs capacity utilization for strict conformal high-only
strict = aggregate[
    aggregate["policy"] == "ConformalHighOnly"
]
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    strict["weighted_capacity_utilization"],
    strict["weighted_recall_at_k"],
    marker="o",
)
for _, row in strict.iterrows():
    ax.annotate(
        f"{int(row['capacity']*100)}%",
        (
            row["weighted_capacity_utilization"],
            row["weighted_recall_at_k"],
        ),
        xytext=(4, 4),
        textcoords="offset points",
    )
ax.set_xlabel("Weighted capacity utilization")
ax.set_ylabel("Weighted Recall@K")
ax.set_title("E6 — Strict conformal eligibility trade-off")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "fig_e6_conformal_capacity_utilization.png",
    dpi=200, bbox_inches="tight"
)
plt.show()



# Interpretation logic

## If RiskTopK dominates

Then the probability ranking is already the most efficient use of scarce intervention slots.  
Conformal uncertainty remains useful for **confidence communication and review routing**, but not necessarily for maximizing captured high-stress cases at fixed \(K\).

## If ConformalHighOnly improves precision but leaves capacity unused

Then singleton-high predictions are a **high-confidence intervention subset**, but are insufficient as the sole institutional triage mechanism.

## If ConformalTriageTopK improves worst-group Recall@K

Then uncertainty-aware prioritization adds operational value beyond pure risk ranking.

## If SourceRecall80 requires >25% intervention

Then a sensitivity target learned from source populations is institutionally infeasible under the tested resource constraints.

## Important claim boundary

E6 evaluates **screening prioritization efficiency**, not clinical benefit.  
`High-stress captured` means correctly prioritized benchmark cases, not successful treatment outcomes.


In [ ]:

manifest = {
    "experiment": "E6 Capacity-Constrained Intervention Policy",
    "dataset_rows_known_group": int(len(e6_df)),
    "excluded_missing_group": int(df[GROUP].isna().sum()),
    "base_model": "group-blind Logistic Regression",
    "features": FEATURES,
    "groups": groups,
    "n_repeats": N_REPEATS,
    "source_calibration_fraction": SOURCE_CAL_FRAC,
    "target_recall_constraint": TARGET_RECALL,
    "mondrian_alpha": ALPHA,
    "capacities": CAPACITIES,
    "capacity_policies": [
        "RiskTopK",
        "ConformalHighOnly",
        "ConformalTriageTopK",
    ],
    "noncapacity_benchmarks": [
        "SourceRecall80",
        "MondrianSingletonHigh90",
    ],
    "seed": SEED,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}

with open(
    RESULT_DIR / "run_manifest.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest, f, indent=2)

print("Saved outputs:")
for p in sorted(RESULT_DIR.glob("*")):
    print("-", p)


# ========== E7 — Robustness, Statistical Inference, and Synthesis ==========

The following cells are preserved from the validated standalone experiment notebook.


# E7 — Robustness, Statistical Inference, and Publication-Ready Synthesis

## Purpose

E7 is the **verification and synthesis layer** for E0–E6. It does not introduce a new prediction algorithm. Instead, it tests whether the principal conclusions remain stable under resampling, data-quality corrections, different capacity assumptions, and alternative conformal confidence levels.

## E7 components

1. **Bootstrap 95% confidence intervals** for key repeated-resampling results.
2. **Paired statistical comparisons** where observations are genuinely paired.
3. **Random-seed sensitivity** of the IID baseline.
4. **Capacity sensitivity** beyond the original 10–25% grid.
5. **Conformal alpha sensitivity** from 80% to 95% nominal coverage.
6. **Predictor-dominance ablation** centered on `Exam_Pressure`.
7. **Original vs plausibility-corrected data sensitivity**.
8. **Publication-ready consolidated tables and figures**.

## Statistical caution

Repeated folds/splits are not independent draws from a population. Therefore:
- bootstrap intervals summarize **resampling stability**, not population sampling uncertainty;
- Wilcoxon tests are used only for paired algorithmic comparisons on the same resampling units;
- LOSO comparisons with only three held-out populations are treated descriptively rather than as formal hypothesis tests.


In [ ]:

from pathlib import Path
import platform, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, recall_score,
    precision_recall_curve
)
from scipy.stats import wilcoxon

SEED = 42
RNG = np.random.default_rng(SEED)
TARGET = "Stress_Level"
GROUP = "Student_Type"

DATA_CANDIDATES = [
    Path("/mnt/data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place student-lifestyle-and-stress-dataset.csv beside this notebook, "
        "inside ../data, or edit DATA_PATH."
    )

RESULT_DIR = Path("../outputs/E7")
RESULT_DIR.mkdir(exist_ok=True)

E0_DIR = Path("../outputs/E0_E2")
E3_DIR = Path("../outputs/E3")
E4_DIR = Path("../outputs/E4")
E5_DIR = Path("../outputs/E5")
E6_DIR = Path("../outputs/E6")

required_dirs = [E0_DIR, E3_DIR, E4_DIR, E5_DIR, E6_DIR]
missing_dirs = [str(p) for p in required_dirs if not p.exists()]
if missing_dirs:
    raise FileNotFoundError(
        "E7 expects E0–E6 outputs in the same working directory. "
        f"Missing: {missing_dirs}. Run E0–E6 first, or use the combined master notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Python :", platform.python_version())
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)
print("Dataset:", DATA_PATH.resolve())
print("Rows   :", len(df))


# Common helpers

In [ ]:

NUMERIC_FEATURES = [
    "Sleep_Hours", "Study_Hours", "Social_Media_Hours",
    "Attendance", "Exam_Pressure", "Family_Support"
]
FULL_GROUPBLIND = NUMERIC_FEATURES + ["Month"]

def make_lr_pipeline(features):
    numeric = [c for c in features if c in NUMERIC_FEATURES]
    categorical = [c for c in features if c == "Month"]

    transformers = []
    if numeric:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric,
        ))
    if categorical:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            categorical,
        ))

    prep = ColumnTransformer(transformers, sparse_threshold=0)
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(
            max_iter=2000, solver="lbfgs", random_state=SEED
        )),
    ])

def bootstrap_mean_ci(values, n_boot=5000, seed=SEED):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    means = x[idx].mean(axis=1)
    return (
        float(x.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )

def paired_bootstrap_diff(a, b, n_boot=5000, seed=SEED):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    d = a[mask] - b[mask]
    if len(d) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    return (
        float(d.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )

def safe_wilcoxon(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 5 or np.allclose(a, b):
        return np.nan, np.nan, len(a)
    try:
        stat, p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
        return float(stat), float(p), len(a)
    except ValueError:
        return np.nan, np.nan, len(a)

def binary_metrics(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    pred = (p >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p),
        "brier": brier_score_loss(y, p),
        "nll": log_loss(y, p, labels=[0, 1]),
        "recall": recall_score(y, pred, zero_division=0),
    }

def joint_strata(frame):
    return (
        frame[GROUP].astype(str)
        + "__Y" + frame[TARGET].astype(int).astype(str)
    )

def select_recall_threshold(y, p, target_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y, p)
    if len(thresholds) == 0:
        return 0.5
    valid = recall[:-1] >= target_recall
    if not np.any(valid):
        return 0.0
    return float(np.max(thresholds[valid]))


# E7.1 — Bootstrap CIs and paired comparisons

In [ ]:

# --- E3 paired calibration comparisons across the same IID outer folds ---
e3 = pd.read_csv(E3_DIR / "e3_iid_fold_metrics.csv")

pair_rows = []
for comparator in ["sigmoid", "isotonic"]:
    for metric in [
        "brier", "nll", "ece_uniform", "ece_quantile",
        "mce_uniform", "mce_quantile"
    ]:
        base = (
            e3[e3["method"] == "uncalibrated"]
            .sort_values(["repeat", "fold"])
        )
        comp = (
            e3[e3["method"] == comparator]
            .sort_values(["repeat", "fold"])
        )

        # Improvement convention:
        # positive => comparator has lower error than uncalibrated.
        diff_mean, ci_lo, ci_hi = paired_bootstrap_diff(
            base[metric].to_numpy(),
            comp[metric].to_numpy(),
            seed=SEED + hash(metric + comparator) % 10000,
        )
        stat, pval, n_pair = safe_wilcoxon(
            base[metric].to_numpy(),
            comp[metric].to_numpy(),
        )
        pair_rows.append({
            "experiment": "E3",
            "comparison": f"{comparator}_vs_uncalibrated",
            "metric": metric,
            "paired_n": n_pair,
            "mean_improvement": diff_mean,
            "bootstrap95_low": ci_lo,
            "bootstrap95_high": ci_hi,
            "wilcoxon_stat": stat,
            "wilcoxon_p": pval,
        })

# --- E5 paired Mondrian vs marginal across heldout × repeat ---
e5 = pd.read_csv(E5_DIR / "e5_conformal_metrics_by_repeat.csv")
e5t = e5[e5["domain"] == "target_loso"].copy()

for alpha in sorted(e5t["alpha"].unique()):
    for metric in [
        "marginal_coverage",
        "coverage_high",
        "decision_coverage",
        "selective_accuracy",
        "actionable_high_stress_recall",
        "false_negative_singleton_rate",
    ]:
        left = (
            e5t[
                (e5t["alpha"] == alpha) &
                (e5t["method"] == "mondrian")
            ]
            .sort_values(["heldout_group", "repeat"])
        )
        right = (
            e5t[
                (e5t["alpha"] == alpha) &
                (e5t["method"] == "marginal")
            ]
            .sort_values(["heldout_group", "repeat"])
        )

        # For false-negative singleton rate, lower is better; flip sign.
        if metric == "false_negative_singleton_rate":
            a, b = right[metric].to_numpy(), left[metric].to_numpy()
            label = "marginal_minus_mondrian"
        else:
            a, b = left[metric].to_numpy(), right[metric].to_numpy()
            label = "mondrian_minus_marginal"

        diff_mean, ci_lo, ci_hi = paired_bootstrap_diff(
            a, b, seed=SEED + int(alpha*1000) + len(metric)
        )
        stat, pval, n_pair = safe_wilcoxon(a, b)
        pair_rows.append({
            "experiment": "E5",
            "comparison": f"{label}_alpha_{alpha}",
            "metric": metric,
            "paired_n": n_pair,
            "mean_improvement": diff_mean,
            "bootstrap95_low": ci_lo,
            "bootstrap95_high": ci_hi,
            "wilcoxon_stat": stat,
            "wilcoxon_p": pval,
        })

# --- E6 strict conformal vs RiskTopK across heldout × repeat ---
e6 = pd.read_csv(E6_DIR / "e6_capacity_results_by_repeat.csv")
for capacity in sorted(e6["capacity"].unique()):
    for metric in ["precision_at_k", "recall_at_k", "lift_at_k"]:
        risk = (
            e6[
                (e6["capacity"] == capacity) &
                (e6["policy"] == "RiskTopK")
            ]
            .sort_values(["heldout_group", "repeat"])
        )
        conf = (
            e6[
                (e6["capacity"] == capacity) &
                (e6["policy"] == "ConformalHighOnly")
            ]
            .sort_values(["heldout_group", "repeat"])
        )

        diff_mean, ci_lo, ci_hi = paired_bootstrap_diff(
            conf[metric].to_numpy(),
            risk[metric].to_numpy(),
            seed=SEED + int(capacity*1000) + len(metric)
        )
        stat, pval, n_pair = safe_wilcoxon(
            conf[metric].to_numpy(),
            risk[metric].to_numpy(),
        )
        pair_rows.append({
            "experiment": "E6",
            "comparison": f"ConformalHighOnly_vs_RiskTopK_K{int(capacity*100)}",
            "metric": metric,
            "paired_n": n_pair,
            "mean_improvement": diff_mean,
            "bootstrap95_low": ci_lo,
            "bootstrap95_high": ci_hi,
            "wilcoxon_stat": stat,
            "wilcoxon_p": pval,
        })

paired_tests = pd.DataFrame(pair_rows)
display(paired_tests.round(6))
paired_tests.to_csv(
    RESULT_DIR / "e7_paired_statistical_comparisons.csv",
    index=False
)


# E7.2 — Random-seed sensitivity of IID baseline

In [ ]:

SEEDS = [1, 7, 21, 42, 84, 123, 202, 314, 512, 777]
seed_rows = []

X = df[FULL_GROUPBLIND].copy()
y = df[TARGET].astype(int).copy()

for seed in SEEDS:
    cv = RepeatedStratifiedKFold(
        n_splits=5, n_repeats=1, random_state=seed
    )
    fold_metrics = []

    for tr, te in cv.split(X, y):
        model = make_lr_pipeline(FULL_GROUPBLIND)
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        fold_metrics.append(binary_metrics(y.iloc[te], p))

    fm = pd.DataFrame(fold_metrics)
    row = {"seed": seed}
    for metric in ["roc_auc", "pr_auc", "brier", "nll", "recall"]:
        row[metric] = fm[metric].mean()
    seed_rows.append(row)

seed_sensitivity = pd.DataFrame(seed_rows)
display(seed_sensitivity.round(6))
seed_sensitivity.to_csv(
    RESULT_DIR / "e7_seed_sensitivity.csv",
    index=False
)

seed_ci_rows = []
for metric in ["roc_auc", "pr_auc", "brier", "nll", "recall"]:
    mean, lo, hi = bootstrap_mean_ci(
        seed_sensitivity[metric],
        seed=SEED + len(metric),
    )
    seed_ci_rows.append({
        "metric": metric,
        "mean_across_seeds": mean,
        "bootstrap95_low": lo,
        "bootstrap95_high": hi,
        "min_seed": seed_sensitivity[metric].min(),
        "max_seed": seed_sensitivity[metric].max(),
        "range": (
            seed_sensitivity[metric].max()
            - seed_sensitivity[metric].min()
        ),
    })

seed_ci = pd.DataFrame(seed_ci_rows)
display(seed_ci.round(6))
seed_ci.to_csv(
    RESULT_DIR / "e7_seed_sensitivity_summary.csv",
    index=False
)


# E7.3 — Predictor-dominance ablation

In [ ]:

ABLATIONS = {
    "FullGroupBlind": FULL_GROUPBLIND,
    "ExamPressureOnly": ["Exam_Pressure"],
    "FullMinusExamPressure": [
        "Sleep_Hours", "Study_Hours", "Social_Media_Hours",
        "Attendance", "Family_Support", "Month"
    ],
    "LifestyleOnly": [
        "Sleep_Hours", "Study_Hours", "Social_Media_Hours",
        "Attendance", "Family_Support"
    ],
}

cv = RepeatedStratifiedKFold(
    n_splits=5, n_repeats=3, random_state=SEED
)

ablation_rows = []
for feature_set, features in ABLATIONS.items():
    X = df[features].copy()
    y = df[TARGET].astype(int).copy()

    for split_id, (tr, te) in enumerate(cv.split(X, y), start=1):
        model = make_lr_pipeline(features)
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        ablation_rows.append({
            "feature_set": feature_set,
            "split_id": split_id,
            **binary_metrics(y.iloc[te], p),
        })

ablation = pd.DataFrame(ablation_rows)
ablation.to_csv(
    RESULT_DIR / "e7_predictor_ablation_by_split.csv",
    index=False
)

ablation_summary_rows = []
for (feature_set, metric), part in (
    ablation.melt(
        id_vars=["feature_set", "split_id"],
        value_vars=["roc_auc", "pr_auc", "brier", "nll", "recall"],
        var_name="metric",
        value_name="value",
    ).groupby(["feature_set", "metric"])
):
    mean, lo, hi = bootstrap_mean_ci(
        part["value"],
        seed=SEED + len(feature_set) + len(metric),
    )
    ablation_summary_rows.append({
        "feature_set": feature_set,
        "metric": metric,
        "mean": mean,
        "bootstrap95_low": lo,
        "bootstrap95_high": hi,
    })

ablation_summary = pd.DataFrame(ablation_summary_rows)
display(
    ablation_summary[
        ablation_summary["metric"].isin(["roc_auc", "brier", "recall"])
    ].round(6)
)
ablation_summary.to_csv(
    RESULT_DIR / "e7_predictor_ablation_summary.csv",
    index=False
)


# E7.4 — Original vs plausibility-corrected data

In [ ]:

# D0: original.
D0 = df.copy()

# D1: flag physically implausible values as missing, then remove exact full duplicates.
D1 = df.copy()
D1.loc[
    D1["Study_Hours"].notna() &
    ~D1["Study_Hours"].between(0, 24),
    "Study_Hours"
] = np.nan
D1.loc[
    D1["Attendance"].notna() &
    ~D1["Attendance"].between(0, 100),
    "Attendance"
] = np.nan
D1 = D1.drop_duplicates().reset_index(drop=True)

quality_versions = {
    "D0_original": D0,
    "D1_plausibility_corrected": D1,
}

quality_rows = []
for version, frame in quality_versions.items():
    X = frame[FULL_GROUPBLIND].copy()
    y = frame[TARGET].astype(int).copy()
    cv = RepeatedStratifiedKFold(
        n_splits=5, n_repeats=3, random_state=SEED
    )

    for split_id, (tr, te) in enumerate(cv.split(X, y), start=1):
        model = make_lr_pipeline(FULL_GROUPBLIND)
        model.fit(X.iloc[tr], y.iloc[tr])
        p = model.predict_proba(X.iloc[te])[:, 1]
        quality_rows.append({
            "version": version,
            "split_id": split_id,
            "n_rows": len(frame),
            **binary_metrics(y.iloc[te], p),
        })

quality = pd.DataFrame(quality_rows)
quality.to_csv(
    RESULT_DIR / "e7_data_quality_sensitivity_by_split.csv",
    index=False
)

quality_summary = (
    quality
    .groupby("version")[
        ["roc_auc", "pr_auc", "brier", "nll", "recall"]
    ]
    .agg(["mean", "std"])
)
display(quality_summary)
quality_summary.to_csv(
    RESULT_DIR / "e7_data_quality_sensitivity_summary.csv"
)


# E7.5 — Extended capacity sensitivity

In [ ]:

e6_pred = pd.read_csv(E6_DIR / "e6_target_row_predictions.csv")
EXTENDED_K = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

cap_rows = []
for (heldout, repeat), part in e6_pred.groupby(
    ["heldout_group", "repeat"]
):
    part = part.copy()
    y = part["y_true"].to_numpy().astype(int)
    p = part["prob_high"].to_numpy()
    prevalence = y.mean()

    for k in EXTENDED_K:
        n_select = max(1, int(np.ceil(k * len(part))))
        order = np.argsort(-p)
        selected = np.zeros(len(part), dtype=bool)
        selected[order[:n_select]] = True

        captured = int(y[selected].sum())
        precision = captured / n_select
        recall = captured / y.sum()

        cap_rows.append({
            "heldout_group": heldout,
            "repeat": repeat,
            "capacity": k,
            "precision_at_k": precision,
            "recall_at_k": recall,
            "lift_at_k": precision / prevalence,
        })

extended_capacity = pd.DataFrame(cap_rows)
extended_capacity.to_csv(
    RESULT_DIR / "e7_extended_capacity_by_repeat.csv",
    index=False
)

cap_summary = (
    extended_capacity
    .groupby("capacity", as_index=False)
    .agg(
        precision_mean=("precision_at_k", "mean"),
        recall_mean=("recall_at_k", "mean"),
        lift_mean=("lift_at_k", "mean"),
        recall_worst_group=(
            "recall_at_k",
            lambda s: np.nan  # filled below
        ),
    )
)

# true worst-group based on group means for each K
wg = (
    extended_capacity
    .groupby(["capacity", "heldout_group"])["recall_at_k"]
    .mean()
    .groupby("capacity")
    .min()
)
cap_summary["recall_worst_group"] = cap_summary["capacity"].map(wg)

display(cap_summary.round(6))
cap_summary.to_csv(
    RESULT_DIR / "e7_extended_capacity_summary.csv",
    index=False
)


# E7.6 — Conformal alpha sensitivity (80%–95%)

In [ ]:

def true_label_scores(y_true, p1):
    y = np.asarray(y_true).astype(int)
    p_true = np.where(y == 1, p1, 1 - p1)
    return 1 - p_true

def qhat(scores, alpha):
    scores = np.sort(np.asarray(scores, dtype=float))
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    k = min(max(k, 1), n)
    return float(scores[k - 1])

def mondrian_q(y, p, alpha):
    y = np.asarray(y).astype(int)
    s = true_label_scores(y, p)
    return {
        0: qhat(s[y == 0], alpha),
        1: qhat(s[y == 1], alpha),
    }

def mondrian_eval(y, p, q):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)
    include0 = p <= q[0]
    include1 = (1 - p) <= q[1]
    set_size = include0.astype(int) + include1.astype(int)
    covered = np.where(y == 0, include0, include1)

    singleton = set_size == 1
    singleton_high = include1 & ~include0
    ambiguous = set_size == 2
    empty = set_size == 0

    return {
        "marginal_coverage": covered.mean(),
        "coverage_high": covered[y == 1].mean(),
        "avg_set_size": set_size.mean(),
        "decision_coverage": singleton.mean(),
        "selective_accuracy": (
            np.where(singleton_high[singleton], 1, 0)
            == y[singleton]
        ).mean() if singleton.any() else np.nan,
        "actionable_high_recall": (
            (singleton_high & (y == 1)).sum() / (y == 1).sum()
        ),
        "high_abstention": (
            ((ambiguous | empty) & (y == 1)).sum() / (y == 1).sum()
        ),
    }

ALPHA_GRID = [0.20, 0.15, 0.10, 0.05]
known = df[df[GROUP].notna()].copy()

alpha_rows = []
for heldout in sorted(known[GROUP].unique()):
    source = known[known[GROUP] != heldout].copy()
    target = known[known[GROUP] == heldout].copy()

    for repeat in range(1, 6):
        seed = SEED + repeat * 101
        train, temp = train_test_split(
            source,
            test_size=0.40,
            random_state=seed,
            stratify=joint_strata(source),
        )
        cal, check = train_test_split(
            temp,
            test_size=0.50,
            random_state=seed + 1000,
            stratify=joint_strata(temp),
        )

        model = make_lr_pipeline(FULL_GROUPBLIND)
        model.fit(train[FULL_GROUPBLIND], train[TARGET].astype(int))

        p_cal = model.predict_proba(cal[FULL_GROUPBLIND])[:, 1]
        p_target = model.predict_proba(target[FULL_GROUPBLIND])[:, 1]
        y_target = target[TARGET].astype(int).to_numpy()

        for alpha in ALPHA_GRID:
            q = mondrian_q(
                cal[TARGET].astype(int).to_numpy(),
                p_cal,
                alpha,
            )
            m = mondrian_eval(y_target, p_target, q)
            alpha_rows.append({
                "heldout_group": heldout,
                "repeat": repeat,
                "alpha": alpha,
                "nominal_coverage": 1-alpha,
                **m,
            })

alpha_sensitivity = pd.DataFrame(alpha_rows)
alpha_sensitivity.to_csv(
    RESULT_DIR / "e7_conformal_alpha_by_repeat.csv",
    index=False
)

alpha_summary = (
    alpha_sensitivity
    .groupby("nominal_coverage", as_index=False)
    .agg(
        marginal_coverage=("marginal_coverage", "mean"),
        coverage_high=("coverage_high", "mean"),
        avg_set_size=("avg_set_size", "mean"),
        decision_coverage=("decision_coverage", "mean"),
        selective_accuracy=("selective_accuracy", "mean"),
        actionable_high_recall=("actionable_high_recall", "mean"),
        high_abstention=("high_abstention", "mean"),
    )
)
display(alpha_summary.round(6))
alpha_summary.to_csv(
    RESULT_DIR / "e7_conformal_alpha_summary.csv",
    index=False
)


# E7.7 — Publication-ready consolidated tables

In [ ]:

# Table 1: dataset audit
t1_missing = pd.read_csv(E0_DIR / "e0_missingness.csv")
t1_validity = pd.read_csv(E0_DIR / "e0_validity_flags.csv")
t1_prev = pd.read_csv(E0_DIR / "e0_prevalence_by_student_type.csv")

table1 = pd.DataFrame({
    "item": [
        "Total observations",
        "High-stress prevalence (%)",
        "Missingness range across predictors (%)",
        "Extra full duplicate rows",
        "Invalid Study_Hours",
        "Invalid Attendance",
        "School high-stress prevalence (%)",
        "College high-stress prevalence (%)",
        "Working-student high-stress prevalence (%)",
    ],
    "value": [
        len(df),
        100*df[TARGET].mean(),
        f"{t1_missing['missing_pct'].min():.2f}–{t1_missing['missing_pct'].max():.2f}",
        int(df.duplicated().sum()),
        int(t1_validity.loc[
            t1_validity["rule"]=="Study_Hours_outside_0_24","invalid_n"
        ].iloc[0]),
        int(t1_validity.loc[
            t1_validity["rule"]=="Attendance_outside_0_100","invalid_n"
        ].iloc[0]),
        float(t1_prev.loc[
            t1_prev["Student_Type_Audit"]=="school","high_stress_pct"
        ].iloc[0]),
        float(t1_prev.loc[
            t1_prev["Student_Type_Audit"]=="college","high_stress_pct"
        ].iloc[0]),
        float(t1_prev.loc[
            t1_prev["Student_Type_Audit"]=="working_student","high_stress_pct"
        ].iloc[0]),
    ]
})
table1.to_csv(RESULT_DIR / "Table1_Dataset_Audit.csv", index=False)

# Table 2: E1/E2 discrimination and shift
e1 = pd.read_csv(E0_DIR / "e1_fold_metrics.csv")
e2 = pd.read_csv(E0_DIR / "e2_loso_metrics.csv")
table2_e1 = (
    e1.groupby("feature_set")[
        ["roc_auc","pr_auc","recall","specificity","brier","nll"]
    ].mean().reset_index()
)
table2_e1.insert(0, "protocol", "IID")
table2_e2 = e2[
    ["heldout_group","roc_auc","pr_auc","recall","specificity","brier","nll"]
].copy()
table2_e2.insert(0, "protocol", "LOSO")
table2_e1.to_csv(RESULT_DIR / "Table2A_IID.csv", index=False)
table2_e2.to_csv(RESULT_DIR / "Table2B_LOSO.csv", index=False)

# Table 3: E3 calibration
table3 = pd.read_csv(E3_DIR / "e3_loso_weighted_worst_summary.csv")
table3.to_csv(RESULT_DIR / "Table3_Calibration_LOSO.csv", index=False)

# Table 4: E4 policy
table4 = pd.read_csv(E4_DIR / "e4_deployable_policy_aggregate.csv")
table4.to_csv(RESULT_DIR / "Table4_Threshold_and_GroupDRO.csv", index=False)

# Table 5: E5 uncertainty
table5 = pd.read_csv(E5_DIR / "e5_target_weighted_worst_summary.csv")
table5.to_csv(RESULT_DIR / "Table5_Conformal_Uncertainty.csv", index=False)

# Table 6: E6 capacity
table6 = pd.read_csv(E6_DIR / "e6_capacity_weighted_worst_summary.csv")
table6.to_csv(RESULT_DIR / "Table6_Capacity_Policy.csv", index=False)

print("Publication tables saved: Table1–Table6")


# E7.8 — Publication-ready figures

In [ ]:

# Figure A — Predictor dominance (ROC-AUC with bootstrap CI)
auc_ab = ablation_summary[ablation_summary["metric"] == "roc_auc"].copy()
auc_ab = auc_ab.sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
xerr = np.vstack([
    auc_ab["mean"] - auc_ab["bootstrap95_low"],
    auc_ab["bootstrap95_high"] - auc_ab["mean"],
])
ax.errorbar(
    auc_ab["mean"],
    np.arange(len(auc_ab)),
    xerr=xerr,
    fmt="o",
    capsize=3,
)
ax.set_yticks(np.arange(len(auc_ab)))
ax.set_yticklabels(auc_ab["feature_set"])
ax.set_xlabel("ROC-AUC")
ax.set_title("E7 — Predictor-dominance ablation")
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "Fig_A_Predictor_Dominance_AUC.png",
    dpi=250, bbox_inches="tight"
)
plt.show()

# Figure B — Extended capacity curve
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    cap_summary["capacity"]*100,
    cap_summary["recall_mean"],
    marker="o",
    label="Weighted Recall@K",
)
ax.plot(
    cap_summary["capacity"]*100,
    cap_summary["precision_mean"],
    marker="o",
    label="Weighted Precision@K",
)
ax.plot(
    cap_summary["capacity"]*100,
    cap_summary["recall_worst_group"],
    marker="o",
    label="Worst-group Recall@K",
)
ax.set_xlabel("Intervention capacity (%)")
ax.set_ylabel("Metric value")
ax.set_title("E7 — Capacity sensitivity")
ax.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "Fig_B_Capacity_Sensitivity.png",
    dpi=250, bbox_inches="tight"
)
plt.show()

# Figure C — Conformal alpha trade-off
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    alpha_summary["nominal_coverage"]*100,
    alpha_summary["coverage_high"],
    marker="o",
    label="High-stress coverage",
)
ax.plot(
    alpha_summary["nominal_coverage"]*100,
    alpha_summary["decision_coverage"],
    marker="o",
    label="Decision coverage",
)
ax.plot(
    alpha_summary["nominal_coverage"]*100,
    alpha_summary["actionable_high_recall"],
    marker="o",
    label="Actionable high-stress recall",
)
ax.set_xlabel("Nominal conformal coverage (%)")
ax.set_ylabel("Metric value")
ax.set_title("E7 — Conformal confidence vs actionability")
ax.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / "Fig_C_Conformal_Alpha_Sensitivity.png",
    dpi=250, bbox_inches="tight"
)
plt.show()



# E7 Decision Gate — Freeze the experimental code

The experimental program is considered sufficiently mature for manuscript preparation if the following hold:

1. **Seed sensitivity is small** relative to the main E1/E2 effects.
2. **Plausibility correction does not materially change conclusions**.
3. `Exam_Pressure` dominance remains evident under repeated CV.
4. Mondrian conformal consistently improves high-stress coverage relative to marginal conformal.
5. Capacity curves show the expected precision–recall trade-off without qualitative instability.
6. No new model is required to explain the central result.

## Recommended manuscript claim hierarchy

### Primary claim
Reliable student-stress screening requires separating:
- discrimination,
- probability calibration,
- uncertainty quantification,
- decision thresholds,
- intervention capacity.

### Secondary claim
Under LOSO population shift, score ranking transfers more robustly than decision and uncertainty policies.

### Operational claim
At fixed capacity, risk ranking is the most efficient prioritization mechanism observed here; conformal inference contributes confidence/abstention semantics rather than a superior ranking.

### Important limitation
The public benchmark is simulated/synthetic-like and cannot support causal or clinical claims. External validation on real institutional data would materially strengthen the study.


In [ ]:

manifest = {
    "experiment": "E7 Robustness, Statistical Inference, and Publication Synthesis",
    "dataset_rows": int(len(df)),
    "seed_sensitivity_seeds": SEEDS,
    "ablation_sets": list(ABLATIONS.keys()),
    "data_quality_versions": list(quality_versions.keys()),
    "extended_capacities": EXTENDED_K,
    "conformal_alpha_grid": ALPHA_GRID,
    "bootstrap_iterations": 5000,
    "paired_test": "Wilcoxon signed-rank on paired resampling units where appropriate",
    "seed": SEED,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}

with open(
    RESULT_DIR / "run_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=2)

print("Saved E7 outputs:")
for p in sorted(RESULT_DIR.glob("*")):
    print("-", p)
